# Syllabus Planner
This is my hackathon project. You give it all your syllabi and it pulls out every deadline and how much each one is worth, puts them in one calendar, shows which weeks are going to be the worst, makes a study plan, and lets you ask questions about your classes.

I got the idea because every syllabus is set up differently. Some have a table of dates, some just say stuff like "due the Tuesday of Week 11", and some have homework every week except certain weeks. It's really easy to miss that three classes have big things due the same week.

**How to run:** in Colab do Runtime > Run all (or Run All in Jupyter). Everything is in this one notebook. If you add an OpenAI key it uses the API, and if you don't it still runs in offline mode using saved results.

Techniques I used:
- **API calls** - all the OpenAI calls go through one wrapper with retries and error messages (2.1)
- **Structured output** - the AI fills in a Pydantic schema instead of returning free text (2.2, 2.4)
- **Prompt engineering** - the extraction prompt has rules for how to write dates, and if the AI makes a mistake it gets a second prompt listing the errors (2.4)
- **RAG** - questions get answered by searching the syllabi, with page numbers (2.6)
- **Agent** - "Ask the Planner" uses function calling with 5 tools (2.9)

The main idea: the AI only reads the syllabus, and Python does all the date and grade math, so the AI can't mess up the numbers.

Layout: Part 1 setup, Part 2 the code, Part 3 sample data, Part 4 demo, Part 5 tests.

### How it fits together
![architecture](attachment:architecture.png)

# Part 1: Setup
This part gets everything ready before any of the real code runs. There are three cells:
1. installs the libraries
2. finds your OpenAI API key
3. sets up the settings the rest of the notebook uses

**Installing libraries.** These are the libraries the project needs:
- `openai` to call the OpenAI API
- `pydantic` for the schemas (the structured format the AI has to fill in)
- `pypdf` to read text out of PDFs
- `reportlab` to make the sample syllabus PDFs in Part 3
- `numpy`, `pandas`, `matplotlib` for the search math, tables, and the chart

If they're already installed (Colab has most of them) this is quick.

In [1]:
# install libraries
%pip install -q openai pydantic pypdf reportlab numpy pandas matplotlib

**Finding the API key.** `load_api_key()` checks three places in order and stops at the first one that has a key:
1. the Colab Secrets panel (the key icon on the left), which is the safest way in Colab
2. a `.env` file in the same folder, for running on your own computer
3. an environment variable that's already set

It prints where it found the key but never prints the key itself. Don't paste your key directly into a cell, because then it gets saved in the notebook. If no key is found, that's fine. The notebook switches to offline mode and still runs everything.

In [2]:
import os
from pathlib import Path


def load_api_key() -> str:
    if os.getenv("OPENAI_API_KEY"):
        return "environment variable"
    try:  # Google Colab secrets
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
        if key:
            os.environ["OPENAI_API_KEY"] = key
            return "Colab secret"
    except Exception:
        pass
    env = Path(".env")
    if env.exists():
        for line in env.read_text().splitlines():
            if line.strip().startswith("OPENAI_API_KEY="):
                os.environ["OPENAI_API_KEY"] = line.split("=", 1)[1].strip().strip('"').strip("'")
                return ".env file"
    return "none found"


print("API key source:", load_api_key())

API key source: Colab secret


**Settings.** The `Config` class keeps all the settings in one place so they're not scattered through the code:
- `SYLLABI_DIR` and `OUTPUT_DIR` are the folders for the sample PDFs and for the files it makes (calendar, CSV, chart)
- `CHAT_MODEL` is the OpenAI model (gpt-4o-mini, cheap and fast) and `EMBED_MODEL` is the model used for search
- `MAX_REPAIR_ROUNDS` is how many times the AI gets to fix its mistakes (2)
- `AGENT_MAX_STEPS` limits how many tool calls the agent can make for one question, so it can't loop forever
- `HOURS_PER_GRADE_POINT`, `DAILY_STUDY_CAP_HOURS` and `PREP_WINDOW_DAYS` control the study plan (how many hours per % of grade, max 4 hours a day, and how many days ahead to start)
- `has_api_key()` is what the rest of the code uses to decide between live and offline mode

At the end it prints whether you're in LIVE or OFFLINE mode.

In [3]:
class Config:
    """Settings."""

    ROOT = Path.cwd()
    SYLLABI_DIR = ROOT / "sample_syllabi"      # sample PDFs go here
    OUTPUT_DIR = ROOT / "outputs"              # calendar, study plan, chart go here

    CHAT_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
    EMBED_MODEL = os.getenv("OPENAI_EMBED_MODEL", "text-embedding-3-small")
    MAX_REPAIR_ROUNDS = int(os.getenv("PLANNER_MAX_REPAIRS", "2"))     # how many times the AI can fix mistakes
    AGENT_MAX_STEPS = int(os.getenv("PLANNER_AGENT_STEPS", "6"))

    HOURS_PER_GRADE_POINT = float(os.getenv("PLANNER_HOURS_PER_POINT", "0.5"))
    DAILY_STUDY_CAP_HOURS = float(os.getenv("PLANNER_DAILY_CAP", "4"))
    PREP_WINDOW_DAYS = int(os.getenv("PLANNER_PREP_WINDOW", "10"))

    @staticmethod
    def has_api_key() -> bool:
        key = os.getenv("OPENAI_API_KEY", "")
        return bool(key) and not key.startswith("sk-your")


config = Config()
print("Mode:", f"LIVE ({config.CHAT_MODEL})" if config.has_api_key() else "OFFLINE (no API key)")

Mode: LIVE (gpt-4o-mini)


# Part 2: The code
This is where all the actual logic is. Running these cells doesn't print anything, it just defines the functions and classes that Part 4 uses. I split it into 10 sections so each one does one job:

| Section | What it does | Technique |
|---|---|---|
| 2.1 | talks to OpenAI | API calls |
| 2.2 | defines the format the AI has to fill in | structured output |
| 2.3 | reads the PDFs | |
| 2.4 | gets the deadlines out and checks for mistakes | structured output + prompt engineering |
| 2.5 | measures accuracy | |
| 2.6 | searches the syllabi to answer questions | RAG |
| 2.7 | crunch weeks, study plan, grade calculator | |
| 2.8 | makes the calendar file and chart | |
| 2.9 | the agent | agents |
| 2.10 | runs everything in order | |

How a syllabus goes through it: **2.3** reads the PDF, then **2.4** sends it to the AI (through **2.1**) which fills in the schema from **2.2**. Then 2.4 turns the dates into real dates and checks everything. The results go to **2.7** for planning and **2.8** for exporting. At the same time, the PDF text gets indexed in **2.6** so you can ask questions, and **2.9** lets the AI use all of this as tools. **2.10** is the class that runs the whole thing.

(You can collapse this part in Colab with the arrow next to the heading.)

## 2.1 Talking to OpenAI (API calls)
Every call to OpenAI in the project goes through this section, so error handling only has to be written once.
- `llm_client()` makes the OpenAI client. If there's no key it raises `LLMUnavailable` with a message instead of a confusing error.
- `_with_retry()` wraps each call. If OpenAI says slow down (rate limit) or the connection drops, it waits and tries again up to 3 times. If the key is wrong (error 401) or the model doesn't exist (error 404) it says exactly that.
- `llm_parse()` is for structured output. You give it a Pydantic schema and it returns a filled-in object, not text. It also checks if the model refused.
- `llm_chat()` is a normal chat call, and can include tools (used by the agent and the Q&A).
- `llm_embed()` turns text into embeddings (lists of numbers) for search. It sends them in batches of 96.

In [4]:
import time
from typing import Any, Type, TypeVar

from pydantic import BaseModel


T = TypeVar("T", bound=BaseModel)


class LLMUnavailable(RuntimeError):
    """Raised when no API key is configured or the API cannot be reached."""


_client = None


def llm_client():
    global _client
    if not config.has_api_key():
        raise LLMUnavailable(
            "OPENAI_API_KEY is not set. Add it to a .env file or run in offline mode."
        )
    if _client is None:
        from openai import OpenAI

        _client = OpenAI(max_retries=3, timeout=90)
    return _client


def _with_retry(fn, attempts: int = 3):
    from openai import APIConnectionError, APIStatusError, RateLimitError

    last: Exception | None = None
    for i in range(attempts):
        try:
            return fn()
        except RateLimitError as e:  # back off and retry
            last = e
            time.sleep(2 * (i + 1))
        except APIConnectionError as e:
            last = e
            time.sleep(1 + i)
        except APIStatusError as e:
            if e.status_code == 401:
                raise LLMUnavailable("OpenAI rejected the API key (401). Check OPENAI_API_KEY.") from e
            if e.status_code == 404:
                raise LLMUnavailable(
                    f"Model '{config.CHAT_MODEL}' not available to this key. Set OPENAI_MODEL."
                ) from e
            raise
    raise LLMUnavailable(f"OpenAI request failed after {attempts} attempts: {last}")


def llm_parse(messages: list[dict[str, Any]], schema: Type[T], model: str | None = None) -> T:
    """Structured output: returns a validated Pydantic object or raises."""

    def call():
        resp = llm_client().chat.completions.parse(
            model=model or config.CHAT_MODEL,
            messages=messages,
            response_format=schema,
        )
        msg = resp.choices[0].message
        if getattr(msg, "refusal", None):
            raise ValueError(f"Model refused: {msg.refusal}")
        if msg.parsed is None:
            raise ValueError("Model returned no parseable content.")
        return msg.parsed

    return _with_retry(call)


def llm_chat(messages: list[dict[str, Any]], tools: list[dict] | None = None, model: str | None = None):
    """Plain chat completion (optionally with tools). Returns the message object."""

    def call():
        kwargs: dict[str, Any] = {"model": model or config.CHAT_MODEL, "messages": messages}
        if tools:
            kwargs["tools"] = tools
        return llm_client().chat.completions.create(**kwargs).choices[0].message

    return _with_retry(call)


def llm_embed(texts: list[str]) -> list[list[float]]:
    def call():
        out: list[list[float]] = []
        for i in range(0, len(texts), 96):
            resp = llm_client().embeddings.create(model=config.EMBED_MODEL, input=texts[i : i + 96])
            out.extend(d.embedding for d in resp.data)
        return out

    return _with_retry(call)

## 2.2 The schemas (structured output)
These Pydantic classes describe exactly what the AI has to return. OpenAI's structured output mode forces the AI to follow the schema, so I always get valid data back instead of text I'd have to parse.

What the AI fills in:
- `CourseExtraction` is the whole syllabus: course code, title, instructor, start date, grade categories, assessments, and late policy.
- `GradeCategory` is something like "Programming Projects, 40%", with how many lowest scores get dropped and a quote from the syllabus as proof.
- `ExtractedAssessment` is one assignment or exam, with a quote as proof.
- `DateSpec` is the important one. It stores the date **the way the syllabus says it**: either an exact date, or a week number + day (like week 11, Tue). The AI never has to convert it.
- `Recurrence` handles repeating stuff, like "every Friday weeks 2-14 except week 11", as one rule instead of 12 separate guesses.

What the rest of the code uses after processing:
- `Deadline` is one real deadline with an actual date and how much of the grade it's worth.
- `Issue` is a problem the checker found (error or warning).
- `CourseReport` is the final result for one class.
- `StudyPlanNarrative` and `WeekAdvice` are for the AI-written study tips in Part 4.

In [5]:
from datetime import date
from enum import Enum
from typing import Literal, Optional

from pydantic import BaseModel, Field

Weekday = Literal["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]


class AssessmentType(str, Enum):
    exam = "exam"
    quiz = "quiz"
    assignment = "assignment"
    project = "project"
    paper = "paper"
    presentation = "presentation"
    lab = "lab"
    other = "other"


# What the LLM returns (strict JSON schema -> every field present, nullable ok)
class DateSpec(BaseModel):
    """Exactly how the syllabus states the date. Fill whichever applies."""

    exact_date: Optional[str] = Field(
        None, description="ISO date YYYY-MM-DD if the syllabus gives a calendar date."
    )
    week: Optional[int] = Field(
        None, description="Course week number if the date is given as 'Week N'."
    )
    weekday: Optional[Weekday] = Field(
        None, description="Day of week if given together with a week number."
    )


class Recurrence(BaseModel):
    """For items that repeat, e.g. 'quiz every Friday in weeks 2-14 except week 9'."""

    weekday: Weekday
    start_week: int
    end_week: int
    skip_weeks: list[int] = Field(default_factory=list)


class GradeCategory(BaseModel):
    name: str = Field(description="Category name exactly as used in the syllabus, e.g. 'Problem Sets'.")
    weight_percent: float = Field(description="Share of the final grade, 0-100.")
    drop_lowest: int = Field(0, description="How many lowest scores are dropped (0 if none).")
    evidence: str = Field(description="Short verbatim quote from the syllabus supporting this weight.")


class ExtractedAssessment(BaseModel):
    title: str = Field(description="Item name, e.g. 'Midterm 1' or 'Problem Set 3'.")
    type: AssessmentType
    category: str = Field(description="Must match one of the grade category names.")
    date: Optional[DateSpec] = Field(None, description="Null when the item recurs (use recurrence).")
    recurrence: Optional[Recurrence] = Field(None, description="Only for repeating items.")
    due_time: Optional[str] = Field(None, description="24h HH:MM if stated, else null.")
    evidence: str = Field(description="Short verbatim quote (<= 25 words) from the syllabus for this item.")


class CourseExtraction(BaseModel):
    course_code: str = Field(description="e.g. 'CSC 316'")
    course_title: str
    instructor: Optional[str] = None
    term_start: Optional[str] = Field(
        None, description="ISO date of the Monday of Week 1, if the syllabus states when the term starts."
    )
    categories: list[GradeCategory]
    assessments: list[ExtractedAssessment]
    late_policy: Optional[str] = Field(None, description="One-sentence summary of the late-work policy.")


# What the rest of the app uses (after deterministic resolution)
class Deadline(BaseModel):
    course: str
    title: str
    type: AssessmentType
    category: str
    due: date
    due_time: Optional[str] = None
    weight_percent: float  # this single item's share of the course grade
    evidence: str
    flags: list[str] = Field(default_factory=list)  # e.g. "evidence not found in syllabus"


class Issue(BaseModel):
    severity: Literal["error", "warning"]
    message: str


class CourseReport(BaseModel):
    """Final, validated output for one syllabus."""

    extraction: CourseExtraction
    deadlines: list[Deadline]
    issues: list[Issue]
    repair_rounds: int = 0
    source: Literal["openai", "reference", "baseline"] = "openai"


# Study-plan narration (LLM structured output)
class WeekAdvice(BaseModel):
    week_of: str = Field(description="ISO date of the Monday of that week.")
    headline: str = Field(description="One punchy line summarising the week.")
    focus: list[str] = Field(description="2-4 concrete things to do that week.")
    risk: Optional[str] = Field(None, description="The biggest risk that week, if any.")


class StudyPlanNarrative(BaseModel):
    overview: str
    weeks: list[WeekAdvice]

## 2.3 Reading the PDFs
Gets the text out of the syllabus files.
- `load_syllabus()` opens one file (PDF, .txt or .md) and keeps the text **page by page**, so later answers can say "page 2". It cleans up extra spaces. If a file is missing, the wrong type, or has almost no text (like a scanned image), it raises `IngestError` with a message explaining why.
- `load_syllabi()` loads a whole folder. If one file is bad it skips it and keeps going, instead of crashing everything.
- `Document` holds a file's pages. `numbered_text()` adds `[[page 1]]` markers before sending it to the AI.
- `chunk_document()` cuts the text into pieces of about 700 characters for the search in 2.6. It splits on paragraphs and never crosses a page, so each chunk knows its page number. `Chunk.cite` makes the citation tag like `[CSC 316 p.1]`.

In [6]:
import re
from dataclasses import dataclass
from pathlib import Path


class IngestError(ValueError):
    pass


@dataclass
class Document:
    name: str          # file stem, used as a stable id
    path: Path
    pages: list[str]   # text per page (1 page for .txt/.md)

    @property
    def text(self) -> str:
        return "\n\n".join(self.pages)

    def numbered_text(self) -> str:
        """Text with page markers, so the LLM (and citations) can refer to pages."""
        return "\n\n".join(f"[[page {i}]]\n{p}" for i, p in enumerate(self.pages, 1))


@dataclass
class Chunk:
    doc: str
    course: str
    page: int
    text: str

    @property
    def cite(self) -> str:
        return f"[{self.course} p.{self.page}]"


def load_syllabus(path: str | Path) -> Document:
    path = Path(path)
    if not path.exists():
        raise IngestError(f"File not found: {path}")
    suffix = path.suffix.lower()
    if suffix == ".pdf":
        from pypdf import PdfReader

        try:
            reader = PdfReader(str(path))
            pages = [(p.extract_text() or "") for p in reader.pages]
        except Exception as e:  # corrupt / encrypted PDFs
            raise IngestError(f"Could not read PDF {path.name}: {e}") from e
    elif suffix in {".txt", ".md"}:
        pages = [path.read_text(encoding="utf-8", errors="replace")]
    else:
        raise IngestError(f"Unsupported file type '{suffix}' for {path.name} (use .pdf, .txt or .md)")

    pages = [_clean_text(p) for p in pages]
    if sum(len(p) for p in pages) < 200:
        raise IngestError(
            f"{path.name} has almost no extractable text - is it a scanned image? Try exporting it as text."
        )
    return Document(name=path.stem, path=path, pages=pages)


def load_syllabi(folder: str | Path) -> list[Document]:
    """Load every syllabus in a folder. A bad file is skipped with a message, not fatal."""
    folder = Path(folder)
    if not folder.is_dir():
        raise IngestError(f"Folder not found: {folder}")
    files = sorted(p for p in folder.iterdir() if p.suffix.lower() in {".pdf", ".txt", ".md"})
    if not files:
        raise IngestError(f"No syllabi (.pdf/.txt/.md) found in {folder}")
    docs = []
    for p in files:
        try:
            docs.append(load_syllabus(p))
        except IngestError as e:
            print(f"[skip] {e}")
    if not docs:
        raise IngestError(f"None of the files in {folder} could be read.")
    return docs


def _clean_text(text: str) -> str:
    text = text.replace(" ", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def chunk_document(doc: Document, course: str, size: int = 700, overlap: int = 120) -> list[Chunk]:
    """Paragraph-aware chunking that never crosses a page boundary (keeps citations exact)."""
    chunks: list[Chunk] = []
    for page_no, page in enumerate(doc.pages, 1):
        paras = [p.strip() for p in re.split(r"\n\s*\n", page) if p.strip()]
        buf = ""
        for para in paras:
            if len(buf) + len(para) + 1 <= size:
                buf = f"{buf}\n{para}".strip()
                continue
            if buf:
                chunks.append(Chunk(doc.name, course, page_no, buf))
                tail = buf[-overlap:].split(" ", 1)[-1] if overlap else ""  # start on a word boundary
                buf = f"{tail}\n{para}".strip()
            else:
                buf = para
            while len(buf) > size:  # very long paragraph: hard split
                chunks.append(Chunk(doc.name, course, page_no, buf[:size]))
                buf = buf[size - overlap :]
        if buf:
            chunks.append(Chunk(doc.name, course, page_no, buf))
    return chunks

## 2.4 Getting the deadlines out + checking for mistakes
This is the main part of the project. It happens in 4 steps:

**Step 1, the AI reads the syllabus** (`_llm_extract`). It sends the syllabus with `SYSTEM_PROMPT`, which has 8 rules. The important ones: write dates exactly how the syllabus says them, put repeating items in as one rule, make sure the weights add to 100, and give a word-for-word quote for everything. The AI returns a `CourseExtraction`.

**Step 2, Python does the dates and math** (`resolve_and_validate`). `week_day()` turns "week 11, Tue" into an actual date using the semester start date. Repeating items get expanded into one deadline per week (skipping the skip weeks). Then each category's weight is split between its items, so 4 projects worth 40% total become 10% each.

**Step 3, checking for mistakes** (also in `resolve_and_validate`). Errors are:
- weights don't add up to 100%
- an assignment uses a category that doesn't exist
- a date lands outside the semester
- a quote isn't actually in the PDF (this catches made-up stuff)

Warnings (shown but not sent back) are things like a TBA date or a duplicate. The quote check is `grounded()`. It lowercases both texts, strips punctuation, and looks for the quote. If it's not an exact match, it slides through the PDF text and allows about an 85% match, because PDFs sometimes break lines weirdly. A made-up quote won't come close.

**Step 4, fixing mistakes** (`analyze_syllabus`). If there are any errors, it sends the AI its own previous answer plus the list of errors and asks for a fixed version. It tries up to 2 times.

`analyze_reference()` is offline mode. It loads the saved correct answer for the sample syllabi instead of calling the AI, but it still goes through steps 2 and 3.

In [7]:
import difflib
import json
import re
from collections import Counter
from datetime import date, timedelta
from pathlib import Path

from pydantic import ValidationError


WEEKDAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
TERM_WEEKS = 18  # 16 teaching weeks + finals

SYSTEM_PROMPT = """You are a meticulous academic registrar. You convert a course syllabus into JSON.

Rules:
1. Report dates EXACTLY as the syllabus states them. If it gives a calendar date, fill
   date.exact_date (YYYY-MM-DD, use the term's year). If it says "Week N, Thursday" or
   "the Thursday of Week N", fill date.week and date.weekday and leave exact_date null.
   Never convert week numbers to calendar dates yourself.
2. Items that repeat (e.g. "a quiz every Friday, Weeks 2-14, except Week 11") are ONE entry
   with a recurrence and date = null. Put excluded weeks in skip_weeks.
3. Items listed together with a shared weight (e.g. "Programming Projects (4) 40%") belong to
   one grade category; list each individual item as its own assessment in that category.
4. Every grade category from the grading section must appear, even if it has no dated items
   (e.g. participation). Category weights must add up to 100.
5. Each assessment.category must match a category name exactly.
6. If an item's date is not stated or is "TBA", set date to null and recurrence to null.
7. evidence fields must be SHORT verbatim quotes copied character-for-character from the text.
8. Do not invent items that are not in the syllabus."""


# 1. LLM extraction (+ repair)
def _llm_extract(doc: Document, feedback: list[str] | None = None, previous: CourseExtraction | None = None):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Syllabus file: {doc.path.name}\n\n{doc.numbered_text()}"},
    ]
    if feedback and previous is not None:
        messages += [
            {"role": "assistant", "content": previous.model_dump_json()},
            {"role": "user", "content": "An automatic checker found these problems with your JSON:\n- "
             + "\n- ".join(feedback)
             + "\nRe-read the syllabus and return a corrected, complete extraction."},
        ]
    return llm_parse(messages, CourseExtraction)


def analyze_syllabus(doc: Document, default_term_start: date, use_llm: bool = True,
            save_dir: Path | None = None) -> CourseReport:
    """Full extraction pipeline for one syllabus, with self-repair."""
    if not use_llm:
        return analyze_reference(doc, default_term_start)

    extraction = _llm_extract(doc)
    rounds = 0
    deadlines, issues = resolve_and_validate(extraction, doc, default_term_start)
    while any(i.severity == "error" for i in issues) and rounds < config.MAX_REPAIR_ROUNDS:
        rounds += 1
        feedback = [i.message for i in issues if i.severity == "error"]
        extraction = _llm_extract(doc, feedback, extraction)
        deadlines, issues = resolve_and_validate(extraction, doc, default_term_start)

    if save_dir:
        save_dir.mkdir(parents=True, exist_ok=True)
        (save_dir / f"{doc.name}.json").write_text(extraction.model_dump_json(indent=2))
    return CourseReport(extraction=extraction, deadlines=deadlines, issues=issues,
                        repair_rounds=rounds, source="openai")


def analyze_reference(doc: Document, default_term_start: date) -> CourseReport:
    """Offline mode: use the hand-verified reference extraction (REFERENCE_EXTRACTIONS, defined below)."""
    data = REFERENCE_EXTRACTIONS.get(doc.name)
    if data is None:
        raise LLMUnavailable(
            f"No API key and no reference extraction for {doc.name}. Set OPENAI_API_KEY to analyse new syllabi."
        )
    try:
        extraction = CourseExtraction.model_validate(data)
    except ValidationError as e:
        raise ValueError(f"Reference extraction for {doc.name} is malformed: {e}") from e
    deadlines, issues = resolve_and_validate(extraction, doc, default_term_start)
    return CourseReport(extraction=extraction, deadlines=deadlines, issues=issues, source="reference")


# 2. Deterministic resolution
def _parse_iso(s: str | None) -> date | None:
    if not s:
        return None
    try:
        return date.fromisoformat(s.strip()[:10])
    except ValueError:
        return None


def week_day(term_start: date, week: int, weekday: str) -> date:
    return term_start + timedelta(weeks=week - 1, days=WEEKDAYS.index(weekday))


def _resolve_date(spec: DateSpec, term_start: date) -> date | None:
    d = _parse_iso(spec.exact_date)
    if d:
        return d
    if spec.week and spec.weekday:
        return week_day(term_start, spec.week, spec.weekday)
    return None


def resolve_and_validate(ex: CourseExtraction, doc: Document | None, default_term_start: date):
    issues: list[Issue] = []
    err = lambda m: issues.append(Issue(severity="error", message=m))  # noqa: E731
    warn = lambda m: issues.append(Issue(severity="warning", message=m))  # noqa: E731

    term_start = _parse_iso(ex.term_start) or default_term_start
    if term_start.weekday() != 0:
        warn(f"term_start {term_start} is not a Monday; snapping to the Monday of that week.")
        term_start -= timedelta(days=term_start.weekday())
    term_end = term_start + timedelta(weeks=TERM_WEEKS)

    # --- grade categories ----------------------------------------------------
    cats = {c.name: c for c in ex.categories}
    total = sum(c.weight_percent for c in ex.categories)
    if abs(total - 100) > 0.5:
        err(f"Grade category weights add up to {total:g}%, not 100%. Check the grading section.")

    # --- expand assessments into dated rows ---------------------------------
    raw: list[tuple[object, date | None, str]] = []  # (assessment, due, title)
    for a in ex.assessments:
        if a.category not in cats:
            err(f"Assessment '{a.title}' uses category '{a.category}', which is not one of: {sorted(cats)}.")
            continue
        if a.recurrence:
            r = a.recurrence
            if not (1 <= r.start_week <= r.end_week <= TERM_WEEKS):
                err(f"'{a.title}' has an impossible recurrence: weeks {r.start_week}-{r.end_week}.")
                continue
            n = 0
            for w in range(r.start_week, r.end_week + 1):
                if w in r.skip_weeks:
                    continue
                n += 1
                raw.append((a, week_day(term_start, w, r.weekday), f"{a.title} #{n} (Week {w})"))
        elif a.date:
            due = _resolve_date(a.date, term_start)
            if due is None:
                err(f"'{a.title}' has an unusable date {a.date.model_dump(exclude_none=True)}.")
                continue
            raw.append((a, due, a.title))
        else:
            warn(f"'{a.title}' has no date in the syllabus (TBA) - it is excluded from the calendar.")
            raw.append((a, None, a.title))

    # --- split category weights across items --------------------------------
    per_cat = Counter(a.category for a, _, _ in raw)
    deadlines: list[Deadline] = []
    source_norm = _norm(doc.text) if doc else None
    for a, due, title in raw:
        cat = cats[a.category]
        flags: list[str] = []
        if source_norm is not None and not grounded(a.evidence, source_norm):
            flags.append("evidence not found in syllabus")
        if due is None:
            continue
        if not (term_start - timedelta(days=7) <= due <= term_end):
            err(f"'{title}' resolves to {due}, outside the term ({term_start} to {term_end}).")
            continue
        deadlines.append(Deadline(
            course=ex.course_code, title=title, type=a.type, category=a.category, due=due,
            due_time=a.due_time, weight_percent=round(cat.weight_percent / per_cat[a.category], 3),
            evidence=a.evidence, flags=flags))

    for c in ex.categories:
        undated_ok = re.search(r"particip|attendance|clicker|engagement", c.name, re.I)
        if per_cat[c.name] == 0 and c.weight_percent >= 10 and not undated_ok:
            warn(f"Category '{c.name}' ({c.weight_percent:g}%) has no dated items - was something missed?")
        if c.drop_lowest and per_cat[c.name] and c.drop_lowest >= per_cat[c.name]:
            err(f"Category '{c.name}' drops {c.drop_lowest} scores but only has {per_cat[c.name]} items.")

    # --- grounding (anti-hallucination) -------------------------------------
    if source_norm is not None:
        for c in ex.categories:
            if not grounded(c.evidence, source_norm):
                err(f"Evidence for category '{c.name}' is not in the syllabus: \"{c.evidence[:80]}\". "
                    "Quote the syllabus verbatim.")
        bad = sorted({d.title.split(' #')[0] for d in deadlines if d.flags})
        for t in bad:
            err(f"Evidence for '{t}' is not in the syllabus. Quote the syllabus verbatim or remove the item.")

    dupes = [k for k, v in Counter((d.title, d.due) for d in deadlines).items() if v > 1]
    for title, due in dupes:
        warn(f"Duplicate item '{title}' on {due}.")

    deadlines.sort(key=lambda d: (d.due, d.due_time or "99:99"))
    return deadlines, issues


# Grounding check
def _norm(s: str) -> str:
    return re.sub(r"[^a-z0-9%]+", " ", s.lower()).strip()


def grounded(quote: str, source_norm: str, threshold: float = 0.85) -> bool:
    """True if `quote` appears (near-)verbatim in the source text."""
    q = _norm(quote)
    if not q:
        return False
    if q in source_norm:
        return True
    qw, sw = q.split(), source_norm.split()
    n = len(qw)
    if n == 0 or len(sw) < n:
        return False
    best = 0.0
    for i in range(0, len(sw) - n + 1):
        window = " ".join(sw[i : i + n])
        r =difflib.SequenceMatcher(None, q, window).ratio()
        if r > best:
            best = r
            if best >= threshold:
                return True
    return False


def report_to_json(report: CourseReport) -> str:
    return json.dumps(json.loads(report.model_dump_json()), indent=2)

## 2.5 Measuring accuracy
To know if it actually works, I compare it to an answer key I made by hand.
- `baseline_extract()` is the no-AI version I compared against. It uses a regex to find dates like "Oct 23" and takes the nearest line with a word like "exam" or "project". It can't handle "Week 11, Tuesday" or "every Friday", which is why it misses most deadlines.
- `score_deadlines()` matches the found deadlines to the answer key (same date, and similar title or same type) and calculates precision (how many found ones are right), recall (how many real ones it found), F1, and the average weight error.
- `load_truth()` gets the answer key for a syllabus from Part 3.

In [8]:
import json
import re
from datetime import date
from pathlib import Path


MONTHS = {m: i for i, m in enumerate(
    ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"], 1)}
DATE_RE = re.compile(r"\b(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\.?\s+(\d{1,2})\b", re.I)
KEYWORDS = re.compile(r"exam|quiz|project|paper|essay|proposal|bibliograph|report|homework|lab|due", re.I)


def baseline_extract(doc: Document, course: str, year: int = 2026) -> list[Deadline]:
    out: list[Deadline] = []
    lines = [l.strip() for l in doc.text.splitlines() if l.strip()]
    for i, line in enumerate(lines):
        for m in DATE_RE.finditer(line):
            try:
                due = date(year, MONTHS[m.group(1)[:3].lower()], int(m.group(2)))
            except ValueError:
                continue
            # label = this line if it has a keyword, else the nearest previous keyword line
            label = None
            for j in range(i, max(i - 3, -1), -1):
                if KEYWORDS.search(lines[j]) and not DATE_RE.fullmatch(lines[j]):
                    label = lines[j]
                    break
            if not label or re.search(r"begin|start", line, re.I):
                continue
            out.append(Deadline(course=course, title=label[:60], type=AssessmentType.other,
                                category="?", due=due, weight_percent=0.0, evidence=line[:80]))
    return out


def _tokens(s: str) -> set[str]:
    return {t for t in re.findall(r"[a-z]+", s.lower()) if len(t) > 2}


def score_deadlines(pred: list[Deadline], truth: list[dict]) -> dict:
    """One-to-one matching on (same due date) and (title overlap or same type)."""
    unmatched = list(pred)
    matched, weight_err = 0, []
    for g in truth:
        gdate, gtok = date.fromisoformat(g["due"]), _tokens(g["title"])
        best = None
        for p in unmatched:
            if p.due != gdate:
                continue
            if gtok & _tokens(p.title) or p.type.value == g["type"]:
                best = p
                break
        if best:
            unmatched.remove(best)
            matched += 1
            if best.weight_percent:
                weight_err.append(abs(best.weight_percent - g["weight"]))
    precision = matched / len(pred) if pred else 0.0
    recall = matched / len(truth) if truth else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"predicted": len(pred), "truth": len(truth), "matched": matched,
            "precision": round(precision, 3), "recall": round(recall, 3), "f1": round(f1, 3),
            "weight_mae": round(sum(weight_err) / len(weight_err), 3) if weight_err else None}


def load_truth(doc_name: str) -> list[dict] | None:
    """Hand-labelled answer key (GROUND_TRUTH, defined in the sample-data section)."""
    return GROUND_TRUTH.get(doc_name)

## 2.6 Searching the syllabi (RAG)
This is for questions like "can I turn in a project late?" The answers are usually in paragraphs, not in the deadline list, so it searches the actual syllabus text.

`SyllabusIndex` does two kinds of search and combines them:
- **Keyword search (BM25)** scores chunks by how many of your words they contain, giving more points for rare words. It's good for exact things like "OWL" or "CSC 316". (`_bm25`)
- **Meaning search (embeddings)** turns each chunk and the question into vectors and compares them. This finds matches even when the wording is different, like "make-up exam" vs "if you miss an exam". It needs an API key.
- **Combining them:** it uses Reciprocal Rank Fusion. Each chunk gets points based on its rank in each list (`1 / (60 + rank)`), so chunks that rank high in both searches end up on top.

Other parts:
- `_tok()` cleans up words before searching, like joining "make-up" into "makeup", removing common words, and making plurals singular.
- `detect_course()` notices if you mention a class ("chem", "CSC 316") and only searches that class.
- `answer_question()` sends the top 4 chunks to the AI with `QA_PROMPT`, which says to only use those chunks, to cite the page after every sentence, and to say "the syllabus doesn't say" if it's not there. Then it checks the citations, and any page the AI cites that it wasn't given gets flagged.
- `_extractive_answer()` is the offline version. It just picks the 1-2 sentences that match your question best.

In [9]:
import math
import re
from collections import Counter

import numpy as np


_TOKEN = re.compile(r"[a-z0-9]+")
STOP = set("the a an of to and or in on for is are be by with at as it this that your you will from "
           "what when how can do does did have has i my me if there any about".split())


def _tok(s: str) -> list[str]:
    """Lowercase tokens, hyphens joined (make-up -> makeup), naive plural stripping."""
    s = re.sub(r"(\w)-(\w)", r"\1\2", s.lower())
    out = []
    for t in _TOKEN.findall(s):
        if t in STOP:
            continue
        if len(t) > 3 and t.endswith("s") and not t.endswith("ss"):
            t = t[:-1]
        out.append(t)
    return out


def detect_course(question: str, courses: list[str]) -> str | None:
    """'is the chem final cumulative?' -> 'CHEM 101' (matches subject prefix or full code)."""
    q = re.sub(r"\s+", "", question.lower())
    for code in courses:
        compact = re.sub(r"\W", "", code.lower())
        subject = re.match(r"[a-z]+", compact)
        if compact in q or (subject and re.search(rf"\b{subject.group()}", question.lower())):
            return code
    return None


class SyllabusIndex:
    def __init__(self, chunks: list[Chunk], use_embeddings: bool | None = None):
        if not chunks:
            raise ValueError("Cannot build an index with no chunks.")
        self.chunks = chunks
        self._docs = [_tok(c.text + " " + c.course) for c in chunks]
        self._df = Counter(t for d in self._docs for t in set(d))
        self._avgdl = sum(map(len, self._docs)) / len(self._docs)
        self.use_embeddings = config.has_api_key() if use_embeddings is None else use_embeddings
        self._emb = None
        if self.use_embeddings:
            try:
                self._emb = self._normalise(np.array(llm_embed([c.text for c in chunks])))
            except LLMUnavailable as e:
                print(f"[rag] embeddings unavailable ({e}); using BM25 only.")
                self.use_embeddings = False

    @staticmethod
    def _normalise(m: np.ndarray) -> np.ndarray:
        return m / np.clip(np.linalg.norm(m, axis=1, keepdims=True), 1e-9, None)

    # -- lexical -----------------------------------------------------------
    def _bm25(self, query: str, k1: float = 1.4, b: float = 0.75) -> np.ndarray:
        q = _tok(query)
        n = len(self._docs)
        scores = np.zeros(n)
        for i, d in enumerate(self._docs):
            tf = Counter(d)
            for t in q:
                if t not in tf:
                    continue
                idf = math.log(1 + (n - self._df[t] + 0.5) / (self._df[t] + 0.5))
                scores[i] += idf * tf[t] * (k1 + 1) / (tf[t] + k1 * (1 - b + b * len(d) / self._avgdl))
        return scores

    # -- hybrid ------------------------------------------------------------
    def search(self, query: str, k: int = 4, course: str | None = None) -> list[tuple[Chunk, float]]:
        rankings = [np.argsort(-self._bm25(query))]
        if self._emb is not None:
            qv = self._normalise(np.array(llm_embed([query])))[0]
            rankings.append(np.argsort(-(self._emb @ qv)))
        fused: dict[int, float] = {}
        for ranking in rankings:  # Reciprocal Rank Fusion
            for rank, idx in enumerate(ranking):
                fused[idx] = fused.get(idx, 0.0) + 1.0 / (60 + rank)
        order = sorted(fused, key=fused.get, reverse=True)
        if course:
            order = [i for i in order if _same_course(self.chunks[i].course, course)] or order
        return [(self.chunks[i], fused[i]) for i in order[:k]]


def _same_course(a: str, b: str) -> bool:
    return re.sub(r"\W", "", a).lower() == re.sub(r"\W", "", b).lower()


QA_PROMPT = """You answer students' questions about their course syllabi.
Use ONLY the numbered context passages. After every factual sentence, cite the passage tag
exactly as shown, e.g. [CSC 316 p.1]. If the context does not contain the answer, say
"The syllabus doesn't say" and suggest asking the instructor. Be concise (<= 4 sentences)."""


def _extractive_answer(question: str, hits: list[tuple[Chunk, float]]) -> dict:
    """Offline fallback: return the 1-2 sentences from the retrieved passages that best match."""
    course_tokens = {t for c, _ in hits for t in _tok(c.course)}
    q = set(_tok(question)) - course_tokens
    need = 2 if len(q) >= 3 else 1
    scored = []
    for rank, (c, _) in enumerate(hits):
        for sent in re.split(r"(?<=[.!?])\s+", c.text):
            lines = sent.split("\n")
            while len(lines) > 1 and len(lines[0].split()) < 5:  # drop table cells / headings
                lines.pop(0)
            sent = " ".join(" ".join(lines).split())
            if len(sent) < 25:
                continue
            if len(sent) > 220:
                sent = sent[:220].rsplit(" ", 1)[0] + "..."
            overlap = len(q & set(_tok(sent)))
            if overlap >= need:
                scored.append((overlap - 0.3 * rank, sent, c.cite))
    scored.sort(key=lambda x: -x[0])
    picked = scored[:2]
    if not picked:
        return {"answer": "The syllabus doesn't seem to say - ask the instructor.", "citations": [],
                "invalid_citations": [], "hits": hits}
    text = " ".join(f"{s} {cite}" for _, s, cite in picked)
    return {"answer": f"(offline, extractive) {text}", "citations": sorted({c for *_, c in picked}),
            "invalid_citations": [], "hits": hits}


def answer_question(index: SyllabusIndex, question: str, course: str | None = None, k: int = 4) -> dict:
    course = course or detect_course(question, sorted({c.course for c in index.chunks}))
    hits = index.search(question, k=k, course=course)
    context = "\n\n".join(f"{c.cite}\n{c.text}" for c, _ in hits)
    valid_tags = {c.cite for c, _ in hits}
    if not config.has_api_key():
        return _extractive_answer(question, hits)
    msg = llm_chat([
        {"role": "system", "content": QA_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ])
    text = msg.content or ""
    cited = set(re.findall(r"\[[^\[\]]+ p\.\d+\]", text))
    return {"answer": text, "citations": sorted(cited & valid_tags),
            "invalid_citations": sorted(cited - valid_tags), "hits": hits}

## 2.7 Crunch weeks, study plan, and grade calculator
All of this is normal Python math. The AI only writes the tips at the end.

**Crunch weeks**
- `workload_by_week()` adds up how much of each class's grade is due each week.
- `crunch_weeks()` flags a week if at least 25% total is due **and** it involves at least 2 classes.

**Study plan** (`schedule()`)
- `prep_hours()` decides how long to study for something. It's about 0.5 hours per 1% of the grade, times 1.5 for exams and 1.3 for projects and papers.
- It plans the earliest deadlines first, putting study time on the emptiest days before each due date, half an hour at a time, and never more than 4 hours in a day.
- If the days before a deadline are already full, it moves the start earlier and adds a warning like "Start CHEM 101 Hour Exam 2 today". If it still doesn't fit, it warns that you're over-committed that week.
- `plan_summary()` adds up the plan by week for the table in Part 4.

**Study tips**
- `narrate()` gives the weekly numbers to the AI, which writes short tips as structured output (`StudyPlanNarrative`). It's told not to change any numbers. Offline it uses a basic template instead.

**Grade calculator**
- `grade_needed()` takes the grades you have so far and a target, and works out the average you need on everything that's left: (target - points you already have) / weight left. For example, with 84% on chem exams, 95% on labs and 100% on homework, getting a 90 needs 90.5% on the rest.

In [10]:
from collections import defaultdict
from dataclasses import dataclass
from datetime import date, timedelta


def monday(d: date) -> date:
    return d - timedelta(days=d.weekday())


# Workload & crunch weeks
def workload_by_week(deadlines: list[Deadline]) -> dict[date, dict[str, float]]:
    """{monday: {course: grade-% due that week}}"""
    out: dict[date, dict[str, float]] = defaultdict(lambda: defaultdict(float))
    for d in deadlines:
        out[monday(d.due)][d.course] += d.weight_percent
    return {w: dict(v) for w, v in sorted(out.items())}


@dataclass
class Crunch:
    week_of: date
    total_weight: float
    courses: list[str]
    big_items: list[Deadline]

    def describe(self) -> str:
        items = ", ".join(f"{d.course} {d.title} ({d.weight_percent:g}%, {d.due:%a %b %d})" for d in self.big_items)
        return f"Week of {self.week_of:%b %d}: {self.total_weight:.0f}% of grades due across {len(self.courses)} courses -> {items}"


def crunch_weeks(deadlines: list[Deadline], threshold: float = 25.0, min_courses: int = 2) -> list[Crunch]:
    """A crunch week has >= `threshold` grade-points due AND involves >= `min_courses` courses."""
    out = []
    for week, per_course in workload_by_week(deadlines).items():
        total = sum(per_course.values())
        if total >= threshold and len(per_course) >= min_courses:
            big = sorted((d for d in deadlines if monday(d.due) == week and d.weight_percent >= 5),
                         key=lambda d: d.due)
            out.append(Crunch(week, total, sorted(per_course), big))
    return out


# Study scheduler (water-filling with a daily cap)
@dataclass
class StudyBlock:
    day: date
    course: str
    target: str
    hours: float


_EFFORT = {AssessmentType.exam: 1.5, AssessmentType.project: 1.3, AssessmentType.paper: 1.3}


def prep_hours(d: Deadline) -> float:
    h = d.weight_percent * config.HOURS_PER_GRADE_POINT * _EFFORT.get(d.type, 1.0)
    return max(0.5, round(h * 2) / 2)  # half-hour granularity


def schedule(deadlines: list[Deadline], today: date, cap: float | None = None,
             window: int | None = None) -> tuple[list[StudyBlock], list[str]]:
    """Spread prep time before each deadline, highest-stakes first, never exceeding `cap` h/day."""
    cap = cap or config.DAILY_STUDY_CAP_HOURS
    window = window or config.PREP_WINDOW_DAYS
    load: dict[date, float] = defaultdict(float)
    blocks: dict[tuple[date, str, str], float] = defaultdict(float)
    warnings: list[str] = []

    shortfall: dict[date, list[tuple[Deadline, float]]] = defaultdict(list)
    upcoming = [d for d in deadlines if d.due > today]
    # Earliest-deadline-first, heavier items first on ties
    for d in sorted(upcoming, key=lambda x: (x.due, -x.weight_percent)):
        need = prep_hours(d)
        span = max(1, min(window, int(need) + 2))
        while True:
            start = max(today, d.due - timedelta(days=span))
            days = [start + timedelta(i) for i in range((d.due - start).days)]
            free = sum(cap - load[x] for x in days)
            if free >= need or start == today:
                break
            span += 2  # stretch the window earlier instead of cramming
        if span > window and d.weight_percent >= 10:
            when = "today" if start == today else f"by {start:%a %b %d}"
            warnings.append(f"Start {d.course} {d.title} {when} - the days before it are crowded.")
        remaining = need
        while remaining > 0:
            candidates = [x for x in days if load[x] + 0.5 <= cap]
            if not candidates:
                shortfall[monday(d.due)].append((d, remaining))
                break
            day = min(candidates, key=lambda x: (load[x], -x.toordinal()))  # emptiest, then latest
            load[day] += 0.5
            blocks[(day, d.course, d.title)] += 0.5
            remaining -= 0.5

    for week, items in sorted(shortfall.items()):
        total = sum(h for _, h in items)
        worst = sorted(items, key=lambda x: -x[0].weight_percent)[:3]
        names = ", ".join(f"{d.course} {d.title}" for d, _ in worst)
        warnings.append(f"Over-committed the week of {week:%b %d}: {total:g}h of prep won't fit under a "
                        f"{cap:g}h/day cap (mostly {names}). Raise the cap, start now, or ask about extensions early.")

    out = [StudyBlock(day, c, t, h) for (day, c, t), h in blocks.items()]
    out.sort(key=lambda b: (b.day, b.course))
    return out, warnings


def plan_summary(blocks: list[StudyBlock], weeks: int = 6, today: date | None = None) -> list[dict]:
    """Per-week rollup used for the narrative prompt and for display."""
    by_week: dict[date, dict] = {}
    for b in blocks:
        w = monday(b.day)
        if today and w >= monday(today) + timedelta(weeks=weeks):
            continue
        row = by_week.setdefault(w, {"week_of": w.isoformat(), "hours": 0.0, "by_target": defaultdict(float)})
        row["hours"] += b.hours
        row["by_target"][f"{b.course} - {b.target}"] += b.hours
    out = []
    for w in sorted(by_week):
        r = by_week[w]
        top = sorted(r["by_target"].items(), key=lambda kv: -kv[1])[:5]
        out.append({"week_of": r["week_of"], "hours": r["hours"], "top": top})
    return out


NARRATE_PROMPT = """You are an upbeat but honest academic coach. Given a student's computed weekly
study plan, deadlines and crunch weeks, write short, specific guidance. Mention real course
names and items. Do not change any numbers or dates you are given."""


def narrate(summary: list[dict], crunches: list[Crunch], today: date) -> StudyPlanNarrative:
    if not config.has_api_key():
        return _template_narrative(summary, crunches)
    facts = {
        "today": today.isoformat(),
        "weeks": [{"week_of": w["week_of"], "study_hours": w["hours"],
                   "top_items": [f"{t} ({h:g}h)" for t, h in w["top"]]} for w in summary],
        "crunch_weeks": [c.describe() for c in crunches if c.week_of >= monday(today)],
    }
    return llm_parse([
        {"role": "system", "content": NARRATE_PROMPT},
        {"role": "user", "content": str(facts)},
    ], StudyPlanNarrative)


def _template_narrative(summary: list[dict], crunches: list[Crunch]) -> StudyPlanNarrative:

    crunch_weeks_set = {c.week_of.isoformat(): c for c in crunches}
    weeks = []
    for w in summary:
        c = crunch_weeks_set.get(w["week_of"])
        weeks.append(WeekAdvice(
            week_of=w["week_of"],
            headline=f"{w['hours']:g} study hours planned" + (" - CRUNCH WEEK" if c else ""),
            focus=[f"{t}: {h:g}h" for t, h in w["top"][:3]],
            risk=c.describe() if c else None))
    return StudyPlanNarrative(overview="(offline template) Plan generated by the scheduler; "
                              "set OPENAI_API_KEY for coaching commentary.", weeks=weeks)


# Grade calculator
LETTERS = [("A", 93), ("A-", 90), ("B+", 87), ("B", 83), ("B-", 80), ("C+", 77), ("C", 73), ("C-", 70), ("D", 60)]


def grade_needed(report: CourseReport, scores: dict[str, float], target: float) -> dict:
    """Average % needed on all *ungraded* categories to finish with `target` %."""
    cats = report.extraction.categories
    resolved, unknown = {}, []
    for key, val in scores.items():
        match = [c for c in cats if c.name.lower().startswith(key.lower().strip())
                 or key.lower().strip() in c.name.lower()]
        if not match:
            unknown.append(key)
            continue
        resolved[match[0].name] = float(val)
    earned = sum(c.weight_percent * resolved[c.name] / 100 for c in cats if c.name in resolved)
    remaining_w = sum(c.weight_percent for c in cats if c.name not in resolved)
    result = {"course": report.extraction.course_code, "target": target,
              "earned_points": round(earned, 2), "remaining_weight": remaining_w,
              "remaining_categories": [c.name for c in cats if c.name not in resolved],
              "unrecognised_categories": unknown}
    if remaining_w == 0:
        result["needed_average"] = None
        result["note"] = f"Everything is graded: final grade {earned:.1f}%."
        return result
    need = (target - earned) / remaining_w * 100
    result["needed_average"] = round(need, 1)
    result["note"] = ("Already locked in." if need <= 0 else
                      "Not reachable even with 100% on everything left." if need > 100 else
                      f"You need an average of {need:.1f}% on the remaining {remaining_w:g}% of the grade.")
    return result

## 2.8 Exporting
Makes the files you can actually use.
- `to_ics()` makes a `.ics` calendar file. It does not connect to your Google account, you download the file and import it into Google, Apple, or Outlook calendar. Each deadline becomes an event with the weight in the title and the syllabus quote in the description. Things worth 10% or more get a reminder 3 days before, smaller stuff gets one 1 day before. The study blocks are added as all-day events.
- `plan_to_csv()` saves the study plan as a spreadsheet (date, class, what to study, hours).
- `workload_chart()` makes the stacked bar chart: one bar per week, colored by class, with a red line at the 25% crunch threshold and a dotted line for today.

In [11]:
import csv
import hashlib
from datetime import date, datetime, timedelta, timezone
from pathlib import Path


def _esc(s: str) -> str:
    return s.replace("\\", "\\\\").replace(";", "\\;").replace(",", "\\,").replace("\n", "\\n")


def _fold(line: str) -> str:
    """Wrap long lines at 75 bytes like the calendar spec says."""
    out, cur = [], ""
    for ch in line:
        limit = 75 if not out else 74
        if len((cur + ch).encode()) > limit:
            out.append(cur)
            cur = ch
        else:
            cur += ch
    out.append(cur)
    return "\r\n ".join(out)


def to_ics(deadlines: list[Deadline], path: Path, blocks: list[StudyBlock] | None = None) -> Path:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    lines = ["BEGIN:VCALENDAR", "VERSION:2.0", "PRODID:-//Syllabus Planner//EN", "CALSCALE:GREGORIAN",
             "X-WR-CALNAME:Syllabus Planner"]
    for d in deadlines:
        uid = hashlib.md5(f"{d.course}{d.title}{d.due}".encode()).hexdigest()
        summary = f"{d.course}: {d.title} ({d.weight_percent:g}%)"
        lines += ["BEGIN:VEVENT", f"UID:{uid}@syllabus-planner", f"DTSTAMP:{stamp}", f"SUMMARY:{_esc(summary)}",
                  f"DESCRIPTION:{_esc('Syllabus: ' + d.evidence)}"]
        if d.due_time:
            h, m = map(int, d.due_time.split(":"))
            start = datetime.combine(d.due, datetime.min.time()).replace(hour=h, minute=m)
            lines += [f"DTSTART:{start:%Y%m%dT%H%M%S}", f"DTEND:{start + timedelta(minutes=30):%Y%m%dT%H%M%S}"]
        else:
            lines += [f"DTSTART;VALUE=DATE:{d.due:%Y%m%d}", f"DTEND;VALUE=DATE:{d.due + timedelta(1):%Y%m%d}"]
        lead = "-P3D" if d.weight_percent >= 10 else "-P1D"
        lines += ["BEGIN:VALARM", "ACTION:DISPLAY", f"DESCRIPTION:{_esc(summary)}", f"TRIGGER:{lead}",
                  "END:VALARM", "END:VEVENT"]
    for b in blocks or []:
        uid = hashlib.md5(f"study{b.day}{b.course}{b.target}".encode()).hexdigest()
        lines += ["BEGIN:VEVENT", f"UID:{uid}@syllabus-planner", f"DTSTAMP:{stamp}",
                  f"SUMMARY:{_esc(f'Study {b.hours:g}h - {b.course} {b.target}')}",
                  f"DTSTART;VALUE=DATE:{b.day:%Y%m%d}", f"DTEND;VALUE=DATE:{b.day + timedelta(1):%Y%m%d}",
                  "TRANSP:TRANSPARENT", "END:VEVENT"]
    lines.append("END:VCALENDAR")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\r\n".join(_fold(l) for l in lines) + "\r\n", newline="")
    return path


def plan_to_csv(blocks: list[StudyBlock], path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["date", "weekday", "course", "prepare_for", "hours"])
        for b in blocks:
            w.writerow([b.day.isoformat(), f"{b.day:%a}", b.course, b.target, b.hours])
    return path


COURSE_COLORS = ["#2a6f97", "#e07a5f", "#81b29a", "#f2cc8f", "#6d597a", "#3d405b"]


def workload_chart(deadlines: list[Deadline], path: Path, today: date | None = None, threshold: float = 25.0):
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    weeks = workload_by_week(deadlines)
    courses = sorted({d.course for d in deadlines})
    xs, w = [], min(weeks)
    while w <= max(weeks):  # include empty weeks so the timeline isn't distorted
        xs.append(w)
        w += timedelta(weeks=1)
    weeks = {x: weeks.get(x, {}) for x in xs}
    fig, ax = plt.subplots(figsize=(11, 4.2), dpi=130)
    bottom = [0.0] * len(xs)
    for i, c in enumerate(courses):
        vals = [weeks[w].get(c, 0.0) for w in xs]
        ax.bar(range(len(xs)), vals, bottom=bottom, color=COURSE_COLORS[i % len(COURSE_COLORS)],
               label=c, width=0.75, edgecolor="white", linewidth=0.6)
        bottom = [b + v for b, v in zip(bottom, vals)]
    ax.axhline(threshold, color="#c1121f", linestyle="--", linewidth=1)
    ax.text(len(xs) - 0.5, threshold + 1, "crunch threshold", color="#c1121f", ha="right", fontsize=8)
    if today:
        tw = monday(today)
        if tw in xs:
            ax.axvline(xs.index(tw), color="#555", linewidth=1, linestyle=":")
            ax.text(xs.index(tw) + 0.1, max(bottom) * 0.95, "today", fontsize=8, color="#555")
    ax.set_xticks(range(len(xs)))
    ax.set_xticklabels([f"{w:%b %d}" for w in xs], rotation=60, fontsize=8)
    ax.set_ylabel("% of course grade due")
    ax.set_title("Where your semester's grade points land, week by week", loc="left", fontsize=11)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=8, ncol=len(courses), loc="upper left")
    fig.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path)
    return fig

## 2.9 Ask the Planner (agent)
The agent can answer questions that need more than one step, like "what should I prioritize this week?" It uses OpenAI function calling: the AI can't see the data directly, but it can call tools, and the tools are just Python functions.

The 5 tools (each one's inputs are a Pydantic class, so OpenAI knows exactly what to send):
- `list_deadlines` gets deadlines, filtered by class, dates, or minimum weight
- `search_syllabus` searches the syllabus text using 2.6
- `grade_needed` runs the grade calculator from 2.7
- `crunch_weeks` gets the crunch weeks from 2.7
- `grade_categories` lists a class's grade breakdown

How `ask()` works:
1. send the question with the system prompt (which says today's date, lists the classes, and says to always use tools instead of guessing)
2. if the AI asks for tools, `call_tool()` runs them and sends the results back
3. repeat until the AI writes a final answer, up to 6 rounds

`call_tool()` catches any error and returns it as a message, so a bad tool call never crashes the loop. The AI just sees the error and can try again.

In [12]:
import json
from datetime import date
from typing import Callable, Optional

import openai
from pydantic import BaseModel, Field


# Tool argument schemas (strict function calling)
class ListDeadlines(BaseModel):
    """List deadlines, optionally filtered by course, date range and minimum weight."""
    course: Optional[str] = Field(None, description="Course code like 'CSC 316', or null for all courses.")
    start: Optional[str] = Field(None, description="ISO start date (inclusive), or null.")
    end: Optional[str] = Field(None, description="ISO end date (inclusive), or null.")
    min_weight: Optional[float] = Field(None, description="Only items worth at least this % of the course grade.")


class SearchSyllabus(BaseModel):
    """Search syllabus text for policies and details (late work, make-ups, what an exam covers...)."""
    query: str
    course: Optional[str] = None


class ScoreEntry(BaseModel):
    category: str = Field(description="Grade category name, e.g. 'Hour Exams'.")
    percent: float = Field(description="The student's current average in that category, 0-100.")


class GradeNeeded(BaseModel):
    """Compute the average needed on all ungraded categories to reach a target course grade."""
    course: str
    target_percent: float
    scores: list[ScoreEntry]


class CrunchWeeks(BaseModel):
    """Find weeks where heavy deadlines from several courses collide."""
    from_date: Optional[str] = Field(None, description="Only weeks on/after this ISO date.")


class Categories(BaseModel):
    """List a course's grade categories and weights."""
    course: str


SYSTEM = """You are Syllabus Planner, a sharp, friendly study assistant for one student.
Today is {today}. The student's courses: {courses}.
Always use tools for dates, weights, grades and policies - never guess. Cite syllabus passages
(e.g. [CHEM 101 p.2]) when you use search results. Give concrete, prioritised advice in <= 150 words."""


class PlannerAgent:
    def __init__(self, reports: list[CourseReport], index: SyllabusIndex, today: date):
        self.reports = {r.extraction.course_code: r for r in reports}
        self.deadlines = sorted((d for r in reports for d in r.deadlines), key=lambda d: d.due)
        self.index = index
        self.today = today
        self._tools: dict[str, tuple[type[BaseModel], Callable]] = {
            "list_deadlines": (ListDeadlines, self._list_deadlines),
            "search_syllabus": (SearchSyllabus, self._search),
            "grade_needed": (GradeNeeded, self._grade_needed),
            "crunch_weeks": (CrunchWeeks, self._crunch),
            "grade_categories": (Categories, self._categories),
        }

    # -- tool implementations ---------------------------------------------
    def _course(self, name: str | None) -> CourseReport | None:
        if not name:
            return None
        key = name.replace(" ", "").lower()
        for code, r in self.reports.items():
            if code.replace(" ", "").lower() in key or key in code.replace(" ", "").lower():
                return r
        return None

    def _list_deadlines(self, a: ListDeadlines):
        rows = self.deadlines
        if a.course:
            r = self._course(a.course)
            if not r:
                return {"error": f"Unknown course '{a.course}'. Known: {list(self.reports)}"}
            rows = [d for d in rows if d.course == r.extraction.course_code]
        if a.start:
            rows = [d for d in rows if d.due >= date.fromisoformat(a.start)]
        if a.end:
            rows = [d for d in rows if d.due <= date.fromisoformat(a.end)]
        if a.min_weight is not None:
            rows = [d for d in rows if d.weight_percent >= a.min_weight]
        return [{"course": d.course, "title": d.title, "due": f"{d.due:%a %Y-%m-%d}", "time": d.due_time,
                 "weight_percent": d.weight_percent, "days_away": (d.due - self.today).days} for d in rows[:40]]

    def _search(self, a: SearchSyllabus):
        return [{"cite": c.cite, "text": c.text} for c, _ in self.index.search(a.query, k=3, course=a.course)]

    def _grade_needed(self, a: GradeNeeded):
        r = self._course(a.course)
        if not r:
            return {"error": f"Unknown course '{a.course}'. Known: {list(self.reports)}"}
        return grade_needed(r, {s.category: s.percent for s in a.scores}, a.target_percent)

    def _crunch(self, a: CrunchWeeks):
        start = date.fromisoformat(a.from_date) if a.from_date else self.today
        return [c.describe() for c in crunch_weeks(self.deadlines) if c.week_of >= monday(start)]

    def _categories(self, a: Categories):
        r = self._course(a.course)
        if not r:
            return {"error": f"Unknown course '{a.course}'. Known: {list(self.reports)}"}
        return [{"name": c.name, "weight_percent": c.weight_percent, "drop_lowest": c.drop_lowest}
                for c in r.extraction.categories]

    # -- public: call a tool directly (used by the offline demo too) --------
    def call_tool(self, name: str, args: dict):
        if name not in self._tools:
            return {"error": f"Unknown tool {name}"}
        schema, fn = self._tools[name]
        try:
            return fn(schema.model_validate(args))
        except Exception as e:  # tools must never crash the loop
            return {"error": f"{type(e).__name__}: {e}"}

    # -- the agent loop --------------------------------------------------------
    def ask(self, question: str, verbose: bool = True) -> dict:
        if not config.has_api_key():
            raise LLMUnavailable("The agent needs OPENAI_API_KEY. Tools can still be called with call_tool().")
        tools = [openai.pydantic_function_tool(schema, name=name) for name, (schema, _) in self._tools.items()]
        messages = [{"role": "system", "content": SYSTEM.format(
                        today=f"{self.today:%A %Y-%m-%d}", courses=", ".join(self.reports))},
                    {"role": "user", "content": question}]
        trace = []
        for _step in range(config.AGENT_MAX_STEPS):
            msg = llm_chat(messages, tools=tools)
            if not msg.tool_calls:
                return {"answer": msg.content, "trace": trace}
            messages.append({"role": "assistant", "content": msg.content, "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls]})
            for tc in msg.tool_calls:
                try:
                    args = json.loads(tc.function.arguments or "{}")
                except json.JSONDecodeError:
                    args = {}
                result = self.call_tool(tc.function.name, args)
                trace.append({"tool": tc.function.name, "args": args})
                if verbose:
                    print(f"  -> {tc.function.name}({json.dumps(args)[:110]})")
                messages.append({"role": "tool", "tool_call_id": tc.id,
                                 "content": json.dumps(result, default=str)[:6000]})
        return {"answer": "I hit my step limit - try a more specific question.", "trace": trace}

## 2.10 Putting it together
The `SyllabusPlanner` class is what Part 4 actually uses. It runs everything in order.
- When you create it you give it the folder of syllabi, the semester start date, and today's date. It figures out LIVE vs OFFLINE mode from whether there's a key.
- `run()` loads every syllabus (2.3), then extracts and checks each one (2.4). If the AI isn't available it falls back to the saved answer, and if a syllabus fails completely it skips it and keeps going. For each class it prints a line with how many deadlines, errors, warnings, and fix rounds. Then it builds the search index (2.6).
- `deadlines` is every deadline from every class, sorted by date.
- `report("CHEM 101")` gets one class's results.
- `ask()` is the quick Q&A, and `agent()` makes the agent.

In [13]:
from datetime import date
from pathlib import Path


class SyllabusPlanner:
    """High-level facade used by the notebook and CLI."""

    def __init__(self, syllabi_dir: str | Path = config.SYLLABI_DIR,
                 term_start: date = date(2026, 8, 17), today: date | None = None,
                 offline: bool | None = None):
        self.syllabi_dir = Path(syllabi_dir)
        self.term_start = term_start
        self.today = today or date.today()
        self.offline = (not config.has_api_key()) if offline is None else offline
        self.docs: list[Document] = []
        self.reports: list[CourseReport] = []
        self.index: SyllabusIndex | None = None

    @property
    def mode(self) -> str:
        return "OFFLINE (reference extractions, BM25 retrieval)" if self.offline else f"LIVE ({config.CHAT_MODEL})"

    def run(self, verbose: bool = True) -> "SyllabusPlanner":
        self.docs = load_syllabi(self.syllabi_dir)
        self.reports = []
        self._doc_names: list[str] = []
        for doc in self.docs:
            try:
                rep = analyze_syllabus(doc, self.term_start, use_llm=not self.offline,
                                      save_dir=config.OUTPUT_DIR / "extractions")
            except LLMUnavailable as e:
                try:
                    rep = analyze_reference(doc, self.term_start)
                    if not self.offline:
                        print(f"[!] {doc.name}: {e} Using the reference extraction instead.")
                except LLMUnavailable:
                    print(f"[skip] {doc.name}: {e}")
                    continue
            except Exception as e:  # model refusal, malformed output after retries, ...
                print(f"[skip] {doc.name}: extraction failed ({type(e).__name__}: {e})")
                continue
            self.reports.append(rep)
            self._doc_names.append(doc.name)
            if verbose:
                errs = sum(i.severity == "error" for i in rep.issues)
                warns = len(rep.issues) - errs
                print(f"  {rep.extraction.course_code:9} {len(rep.deadlines):3} deadlines | "
                      f"{errs} errors, {warns} warnings | repair rounds: {rep.repair_rounds} | source: {rep.source}")
        if not self.reports:
            raise ValueError("No syllabus could be analysed (see messages above).")
        by_name = {d.name: d for d in self.docs}
        self.docs = [by_name[n] for n in self._doc_names]  # keep docs aligned with reports
        chunks = [c for doc, rep in zip(self.docs, self.reports)
                  for c in chunk_document(doc, rep.extraction.course_code)]
        self.index = SyllabusIndex(chunks, use_embeddings=not self.offline)
        return self

    @property
    def deadlines(self) -> list[Deadline]:
        return sorted((d for r in self.reports for d in r.deadlines), key=lambda d: (d.due, d.course))

    def report(self, course: str) -> CourseReport:
        key = course.replace(" ", "").lower()
        for r in self.reports:
            if r.extraction.course_code.replace(" ", "").lower() == key:
                return r
        raise KeyError(f"No course {course}. Have: {[r.extraction.course_code for r in self.reports]}")

    def agent(self) -> PlannerAgent:
        return PlannerAgent(self.reports, self.index, self.today)

    def ask(self, question: str, course: str | None = None) -> dict:
        return answer_question(self.index, question, course=course)

# Part 3: Sample data
I made three fake syllabi to test it, each written in a different style:
- **CSC 316** has a table of dates and a quiz every Friday except Week 11
- **HIST 207** is all paragraphs with dates like "the Tuesday of Week 11"
- **CHEM 101** is a mix, with weekly homework and a lab practical that's TBA

This cell makes them into PDFs in a `sample_syllabi` folder.

In [14]:
from pathlib import Path

from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import PageBreak, Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle

OUT = config.SYLLABI_DIR
styles = getSampleStyleSheet()
H1, H2, P = styles["Title"], styles["Heading2"], styles["BodyText"]


def table(rows, widths):
    t = Table(rows, colWidths=widths)
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#dde3ea")),
        ("GRID", (0, 0), (-1, -1), 0.4, colors.grey),
        ("FONTSIZE", (0, 0), (-1, -1), 9),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
    ]))
    return t


def build(name, story):
    SimpleDocTemplate(str(OUT / name), pagesize=letter, title=name).build(story)


def csc316():
    s = [Paragraph("CSC 316: Data Structures and Algorithms", H1),
         Paragraph("Fall 2026 &middot; Dr. Priya Raman &middot; MWF 10:40-11:30, EB2 1231", P),
         Paragraph("Classes begin Monday, August 17, 2026. Office hours: Tuesdays 2-4 PM, EB2 3254.", P),
         Spacer(1, 8),
         Paragraph("Course Description", H2),
         Paragraph("Abstract data types, lists, stacks, queues, trees, heaps, hashing, graphs and the "
                   "analysis of algorithms that operate on them. Programming is done in Java.", P),
         Paragraph("Grading", H2),
         table([["Component", "Weight", "Notes"],
                ["Programming Projects (4)", "40%", "10% each"],
                ["Weekly Quizzes", "10%", "Lowest two quiz scores are dropped"],
                ["Midterm Exam", "20%", "Evening exam"],
                ["Final Exam", "25%", "Cumulative"],
                ["Participation", "5%", "In-class exercises"]], [170, 60, 230]),
         Spacer(1, 8),
         Paragraph("Weekly quizzes are taken on Gradescope every Friday from Week 2 through Week 14, "
                   "except Week 11 (midterm week). Quizzes close at 11:59 PM.", P),
         Paragraph("Late Policy", H2),
         Paragraph("Projects may be submitted up to 48 hours late for a 10% penalty per day. "
                   "Quizzes cannot be made up.", P),
         PageBreak(),
         Paragraph("Major Deadlines", H2),
         table([["Item", "Due", "Time"],
                ["Project 1: Linked Lists", "Fri Sep 11", "11:59 PM"],
                ["Project 2: Binary Search Trees", "Fri Oct 2", "11:59 PM"],
                ["Project 3: Hash Maps", "Fri Oct 23", "11:59 PM"],
                ["Midterm Exam (Weeks 1-10)", "Wed Oct 28", "7:00 PM, Dabney 222"],
                ["Project 4: Graph Algorithms", "Fri Nov 20", "11:59 PM"],
                ["Final Exam", "Thu Dec 10", "8:00 AM"]], [220, 110, 130]),
         Spacer(1, 8),
         Paragraph("Academic Integrity", H2),
         Paragraph("All projects are individual work. Code similarity tools are used on every submission.", P)]
    build("CSC316_Data_Structures.pdf", s)


def hist207():
    s = [Paragraph("HIST 207: Rivers, Railways and Rebellion", H1),
         Paragraph("Latin America, 1800-1950 &middot; Fall 2026 &middot; Prof. Martin Echeverria", P),
         Paragraph("The semester begins Monday, August 17, 2026. We meet Tuesdays and Thursdays, 1:30-2:45.", P),
         Paragraph("How You Will Be Evaluated", H2),
         Paragraph("Your grade has several parts. Reading responses make up 20% of the grade; your two "
                   "lowest responses are dropped. A short map quiz counts for 5%. The midterm essay is worth "
                   "25%. The archival research project is the heart of the course and is graded in three "
                   "stages: the proposal (5%), the annotated bibliography (10%) and the final paper (25%). "
                   "Participation in discussion counts for the remaining 10%.", P),
         Paragraph("Reading responses of about 300 words are due on Moodle before class every Tuesday, "
                   "Weeks 2 through 13.", P),
         Paragraph("The map quiz will be given in class on the Thursday of Week 3.", P),
         Paragraph("The midterm essay prompt is released in Week 9; the essay is due on the Tuesday of Week 11 "
                   "by 9:00 AM.", P),
         PageBreak(),
         Paragraph("The Archival Project", H2),
         Paragraph("You will choose a primary-source collection from the library's digital archive. "
                   "Your one-page proposal is due Friday, September 25. The annotated bibliography of at "
                   "least eight sources is due on the Friday of Week 13. The final paper (3,000-3,500 words) "
                   "is due Friday, December 11 at 5:00 PM.", P),
         Paragraph("Late Work", H2),
         Paragraph("Late essays and project stages lose one third of a letter grade per day. Reading "
                   "responses are not accepted late, which is why the two lowest are dropped.", P),
         Paragraph("Accommodations", H2),
         Paragraph("Students with documented accommodations should contact me in the first two weeks.", P)]
    build("HIST207_Latin_America.pdf", s)


def chem101():
    s = [Paragraph("CHEM 101: General Chemistry I", H1),
         Paragraph("Fall 2026 &middot; Dr. Alan Okafor &middot; Lecture MWF 9:10, Lab weekly", P),
         Paragraph("Week 1 starts Monday, August 17, 2026.", P),
         Paragraph("Course Grade", H2),
         table([["Category", "Weight"],
                ["Hour Exams (3, equally weighted)", "36%"],
                ["Final Exam (cumulative)", "24%"],
                ["Lab Reports", "20%"],
                ["Online Homework (OWL)", "10%"],
                ["Lab Practical", "5%"],
                ["Clicker Participation", "5%"]], [260, 80]),
         Spacer(1, 6),
         Paragraph("Your single lowest lab report score is dropped.", P),
         Paragraph("Assignments", H2),
         Paragraph("OWL homework sets are due every Sunday at 11:59 PM, Weeks 1 through 15.", P),
         Paragraph("Lab reports are due Wednesdays by 11:59 PM, Weeks 3 through 14, except Week 11.", P),
         Paragraph("The lab practical date will be announced in lab (TBA).", P),
         PageBreak(),
         Paragraph("Exam Schedule", H2),
         table([["Exam", "When", "Coverage"],
                ["Hour Exam 1", "Week 5, Thursday", "Ch. 1-3"],
                ["Hour Exam 2", "Week 11, Thursday", "Ch. 4-6"],
                ["Hour Exam 3", "Week 14, Thursday", "Ch. 7-9"],
                ["Final Exam", "Monday, December 14, 12:00 PM", "Cumulative"]], [110, 200, 120]),
         Spacer(1, 8),
         Paragraph("Missed Exams", H2),
         Paragraph("There are no make-up hour exams. With a documented excuse, the final exam score "
                   "replaces one missed hour exam. Late lab reports lose 20% per day.", P)]
    build("CHEM101_General_Chemistry.pdf", s)


OUT.mkdir(parents=True, exist_ok=True)
csc316(); hist207(); chem101()
print("Sample syllabi:", sorted(p.name for p in OUT.glob("*.pdf")))

Sample syllabi: ['CHEM101_General_Chemistry.pdf', 'CSC316_Data_Structures.pdf', 'HIST207_Latin_America.pdf']


These are the correct extractions for the three samples. Offline mode uses them so the notebook still works without a key.

In [15]:
import json

REFERENCE_EXTRACTIONS = json.loads(r'''
{
 "CHEM101_General_Chemistry": {
  "course_code": "CHEM 101",
  "course_title": "General Chemistry I",
  "instructor": "Dr. Alan Okafor",
  "term_start": "2026-08-17",
  "categories": [
   {
    "name": "Hour Exams",
    "weight_percent": 36,
    "drop_lowest": 0,
    "evidence": "Hour Exams (3, equally weighted) 36%"
   },
   {
    "name": "Final Exam",
    "weight_percent": 24,
    "drop_lowest": 0,
    "evidence": "Final Exam (cumulative) 24%"
   },
   {
    "name": "Lab Reports",
    "weight_percent": 20,
    "drop_lowest": 1,
    "evidence": "Your single lowest lab report score is dropped"
   },
   {
    "name": "Online Homework (OWL)",
    "weight_percent": 10,
    "drop_lowest": 0,
    "evidence": "Online Homework (OWL) 10%"
   },
   {
    "name": "Lab Practical",
    "weight_percent": 5,
    "drop_lowest": 0,
    "evidence": "Lab Practical 5%"
   },
   {
    "name": "Clicker Participation",
    "weight_percent": 5,
    "drop_lowest": 0,
    "evidence": "Clicker Participation 5%"
   }
  ],
  "assessments": [
   {
    "title": "OWL Homework",
    "type": "assignment",
    "category": "Online Homework (OWL)",
    "date": null,
    "recurrence": {
     "weekday": "Sun",
     "start_week": 1,
     "end_week": 15,
     "skip_weeks": []
    },
    "due_time": "23:59",
    "evidence": "OWL homework sets are due every Sunday at 11:59 PM, Weeks 1 through 15"
   },
   {
    "title": "Lab Report",
    "type": "lab",
    "category": "Lab Reports",
    "date": null,
    "recurrence": {
     "weekday": "Wed",
     "start_week": 3,
     "end_week": 14,
     "skip_weeks": [
      11
     ]
    },
    "due_time": "23:59",
    "evidence": "Lab reports are due Wednesdays by 11:59 PM, Weeks 3 through 14, except Week 11"
   },
   {
    "title": "Lab Practical",
    "type": "exam",
    "category": "Lab Practical",
    "date": null,
    "recurrence": null,
    "due_time": null,
    "evidence": "The lab practical date will be announced in lab (TBA)"
   },
   {
    "title": "Hour Exam 1",
    "type": "exam",
    "category": "Hour Exams",
    "date": {
     "exact_date": null,
     "week": 5,
     "weekday": "Thu"
    },
    "recurrence": null,
    "due_time": null,
    "evidence": "Hour Exam 1 Week 5, Thursday"
   },
   {
    "title": "Hour Exam 2",
    "type": "exam",
    "category": "Hour Exams",
    "date": {
     "exact_date": null,
     "week": 11,
     "weekday": "Thu"
    },
    "recurrence": null,
    "due_time": null,
    "evidence": "Hour Exam 2 Week 11, Thursday"
   },
   {
    "title": "Hour Exam 3",
    "type": "exam",
    "category": "Hour Exams",
    "date": {
     "exact_date": null,
     "week": 14,
     "weekday": "Thu"
    },
    "recurrence": null,
    "due_time": null,
    "evidence": "Hour Exam 3 Week 14, Thursday"
   },
   {
    "title": "Final Exam",
    "type": "exam",
    "category": "Final Exam",
    "date": {
     "exact_date": "2026-12-14",
     "week": null,
     "weekday": null
    },
    "recurrence": null,
    "due_time": "12:00",
    "evidence": "Final Exam Monday, December 14, 12:00 PM"
   }
  ],
  "late_policy": "Late lab reports lose 20% per day; no make-up hour exams (final can replace one excused miss)."
 },
 "CSC316_Data_Structures": {
  "course_code": "CSC 316",
  "course_title": "Data Structures and Algorithms",
  "instructor": "Dr. Priya Raman",
  "term_start": "2026-08-17",
  "categories": [
   {
    "name": "Programming Projects",
    "weight_percent": 40,
    "drop_lowest": 0,
    "evidence": "Programming Projects (4) 40% 10% each"
   },
   {
    "name": "Weekly Quizzes",
    "weight_percent": 10,
    "drop_lowest": 2,
    "evidence": "Lowest two quiz scores are dropped"
   },
   {
    "name": "Midterm Exam",
    "weight_percent": 20,
    "drop_lowest": 0,
    "evidence": "Midterm Exam 20%"
   },
   {
    "name": "Final Exam",
    "weight_percent": 25,
    "drop_lowest": 0,
    "evidence": "Final Exam 25% Cumulative"
   },
   {
    "name": "Participation",
    "weight_percent": 5,
    "drop_lowest": 0,
    "evidence": "Participation 5%"
   }
  ],
  "assessments": [
   {
    "title": "Project 1: Linked Lists",
    "type": "project",
    "category": "Programming Projects",
    "date": {
     "exact_date": "2026-09-11",
     "week": null,
     "weekday": null
    },
    "recurrence": null,
    "due_time": "23:59",
    "evidence": "Project 1: Linked Lists Fri Sep 11"
   },
   {
    "title": "Project 2: Binary Search Trees",
    "type": "project",
    "category": "Programming Projects",
    "date": {
     "exact_date": "2026-10-02",
     "week": null,
     "weekday": null
    },
    "recurrence": null,
    "due_time": "23:59",
    "evidence": "Project 2: Binary Search Trees Fri Oct 2"
   },
   {
    "title": "Project 3: Hash Maps",
    "type": "project",
    "category": "Programming Projects",
    "date": {
     "exact_date": "2026-10-23",
     "week": null,
     "weekday": null
    },
    "recurrence": null,
    "due_time": "23:59",
    "evidence": "Project 3: Hash Maps Fri Oct 23"
   },
   {
    "title": "Project 4: Graph Algorithms",
    "type": "project",
    "category": "Programming Projects",
    "date": {
     "exact_date": "2026-11-20",
     "week": null,
     "weekday": null
    },
    "recurrence": null,
    "due_time": "23:59",
    "evidence": "Project 4: Graph Algorithms Fri Nov 20"
   },
   {
    "title": "Weekly Quiz",
    "type": "quiz",
    "category": "Weekly Quizzes",
    "date": null,
    "recurrence": {
     "weekday": "Fri",
     "start_week": 2,
     "end_week": 14,
     "skip_weeks": [
      11
     ]
    },
    "due_time": "23:59",
    "evidence": "every Friday from Week 2 through Week 14, except Week 11"
   },
   {
    "title": "Midterm Exam",
    "type": "exam",
    "category": "Midterm Exam",
    "date": {
     "exact_date": "2026-10-28",
     "week": null,
     "weekday": null
    },
    "recurrence": null,
    "due_time": "19:00",
    "evidence": "Midterm Exam (Weeks 1-10) Wed Oct 28 7:00 PM"
   },
   {
    "title": "Final Exam",
    "type": "exam",
    "category": "Final Exam",
    "date": {
     "exact_date": "2026-12-10",
     "week": null,
     "weekday": null
    },
    "recurrence": null,
    "due_time": "08:00",
    "evidence": "Final Exam Thu Dec 10 8:00 AM"
   }
  ],
  "late_policy": "Projects can be up to 48 hours late at 10% per day; quizzes cannot be made up."
 },
 "HIST207_Latin_America": {
  "course_code": "HIST 207",
  "course_title": "Rivers, Railways and Rebellion: Latin America, 1800-1950",
  "instructor": "Prof. Martin Echeverria",
  "term_start": "2026-08-17",
  "categories": [
   {
    "name": "Reading Responses",
    "weight_percent": 20,
    "drop_lowest": 2,
    "evidence": "Reading responses make up 20% of the grade"
   },
   {
    "name": "Map Quiz",
    "weight_percent": 5,
    "drop_lowest": 0,
    "evidence": "A short map quiz counts for 5%"
   },
   {
    "name": "Midterm Essay",
    "weight_percent": 25,
    "drop_lowest": 0,
    "evidence": "The midterm essay is worth 25%"
   },
   {
    "name": "Project Proposal",
    "weight_percent": 5,
    "drop_lowest": 0,
    "evidence": "the proposal (5%)"
   },
   {
    "name": "Annotated Bibliography",
    "weight_percent": 10,
    "drop_lowest": 0,
    "evidence": "the annotated bibliography (10%)"
   },
   {
    "name": "Final Paper",
    "weight_percent": 25,
    "drop_lowest": 0,
    "evidence": "the final paper (25%)"
   },
   {
    "name": "Participation",
    "weight_percent": 10,
    "drop_lowest": 0,
    "evidence": "Participation in discussion counts for the remaining 10%"
   }
  ],
  "assessments": [
   {
    "title": "Reading Response",
    "type": "assignment",
    "category": "Reading Responses",
    "date": null,
    "recurrence": {
     "weekday": "Tue",
     "start_week": 2,
     "end_week": 13,
     "skip_weeks": []
    },
    "due_time": null,
    "evidence": "due on Moodle before class every Tuesday, Weeks 2 through 13"
   },
   {
    "title": "Map Quiz",
    "type": "quiz",
    "category": "Map Quiz",
    "date": {
     "exact_date": null,
     "week": 3,
     "weekday": "Thu"
    },
    "recurrence": null,
    "due_time": null,
    "evidence": "The map quiz will be given in class on the Thursday of Week 3"
   },
   {
    "title": "Midterm Essay",
    "type": "paper",
    "category": "Midterm Essay",
    "date": {
     "exact_date": null,
     "week": 11,
     "weekday": "Tue"
    },
    "recurrence": null,
    "due_time": "09:00",
    "evidence": "the essay is due on the Tuesday of Week 11 by 9:00 AM"
   },
   {
    "title": "Project Proposal",
    "type": "assignment",
    "category": "Project Proposal",
    "date": {
     "exact_date": "2026-09-25",
     "week": null,
     "weekday": null
    },
    "recurrence": null,
    "due_time": null,
    "evidence": "Your one-page proposal is due Friday, September 25"
   },
   {
    "title": "Annotated Bibliography",
    "type": "assignment",
    "category": "Annotated Bibliography",
    "date": {
     "exact_date": null,
     "week": 13,
     "weekday": "Fri"
    },
    "recurrence": null,
    "due_time": null,
    "evidence": "The annotated bibliography of at least eight sources is due on the Friday of Week 13"
   },
   {
    "title": "Final Paper",
    "type": "paper",
    "category": "Final Paper",
    "date": {
     "exact_date": "2026-12-11",
     "week": null,
     "weekday": null
    },
    "recurrence": null,
    "due_time": "17:00",
    "evidence": "The final paper (3,000-3,500 words) is due Friday, December 11 at 5:00 PM"
   }
  ],
  "late_policy": "Late essays and project stages lose a third of a letter grade per day; reading responses are not accepted late."
 }
}
''')
print({k: len(v['assessments']) for k, v in REFERENCE_EXTRACTIONS.items()}, 'items')

{'CHEM101_General_Chemistry': 7, 'CSC316_Data_Structures': 7, 'HIST207_Latin_America': 6} items


Answer key: I went through all 65 deadlines by hand to check accuracy against.

In [16]:
GROUND_TRUTH = json.loads(r'''{
 "CHEM101_General_Chemistry": [
  {"title": "Hour Exam 1", "due": "2026-09-17", "weight": 12, "type": "exam"},
  {"title": "Hour Exam 2", "due": "2026-10-29", "weight": 12, "type": "exam"},
  {"title": "Hour Exam 3", "due": "2026-11-19", "weight": 12, "type": "exam"},
  {"title": "Final Exam", "due": "2026-12-14", "weight": 24, "type": "exam"},
  {"title": "OWL Homework", "due": "2026-08-23", "weight": 0.6666666666666666, "type": "assignment"},
  {"title": "OWL Homework", "due": "2026-08-30", "weight": 0.6666666666666666, "type": "assignment"},
  {"title": "OWL Homework", "due": "2026-09-06", "weight": 0.6666666666666666, "type": "assignment"},
  {"title": "OWL Homework", "due": "2026-09-13", "weight": 0.6666666666666666, "type": "assignment"},
  {"title": "OWL Homework", "due": "2026-09-20", "weight": 0.6666666666666666, "type": "assignment"},
  {"title": "OWL Homework", "due": "2026-09-27", "weight": 0.6666666666666666, "type": "assignment"},
  {"title": "OWL Homework", "due": "2026-10-04", "weight": 0.6666666666666666, "type": "assignment"},
  {"title": "OWL Homework", "due": "2026-10-11", "weight": 0.6666666666666666, "type": "assignment"},
  {"title": "OWL Homework", "due": "2026-10-18", "weight": 0.6666666666666666, "type": "assignment"},
  {"title": "OWL Homework", "due": "2026-10-25", "weight": 0.6666666666666666, "type": "assignment"},
  {"title": "OWL Homework", "due": "2026-11-01", "weight": 0.6666666666666666, "type": "assignment"},
  {"title": "OWL Homework", "due": "2026-11-08", "weight": 0.6666666666666666, "type": "assignment"},
  {"title": "OWL Homework", "due": "2026-11-15", "weight": 0.6666666666666666, "type": "assignment"},
  {"title": "OWL Homework", "due": "2026-11-22", "weight": 0.6666666666666666, "type": "assignment"},
  {"title": "OWL Homework", "due": "2026-11-29", "weight": 0.6666666666666666, "type": "assignment"},
  {"title": "Lab Report", "due": "2026-09-02", "weight": 1.8181818181818181, "type": "lab"},
  {"title": "Lab Report", "due": "2026-09-09", "weight": 1.8181818181818181, "type": "lab"},
  {"title": "Lab Report", "due": "2026-09-16", "weight": 1.8181818181818181, "type": "lab"},
  {"title": "Lab Report", "due": "2026-09-23", "weight": 1.8181818181818181, "type": "lab"},
  {"title": "Lab Report", "due": "2026-09-30", "weight": 1.8181818181818181, "type": "lab"},
  {"title": "Lab Report", "due": "2026-10-07", "weight": 1.8181818181818181, "type": "lab"},
  {"title": "Lab Report", "due": "2026-10-14", "weight": 1.8181818181818181, "type": "lab"},
  {"title": "Lab Report", "due": "2026-10-21", "weight": 1.8181818181818181, "type": "lab"},
  {"title": "Lab Report", "due": "2026-11-04", "weight": 1.8181818181818181, "type": "lab"},
  {"title": "Lab Report", "due": "2026-11-11", "weight": 1.8181818181818181, "type": "lab"},
  {"title": "Lab Report", "due": "2026-11-18", "weight": 1.8181818181818181, "type": "lab"}
 ],
 "CSC316_Data_Structures": [
  {"title": "Project 1: Linked Lists", "due": "2026-09-11", "weight": 10, "type": "project"},
  {"title": "Project 2: Binary Search Trees", "due": "2026-10-02", "weight": 10, "type": "project"},
  {"title": "Project 3: Hash Maps", "due": "2026-10-23", "weight": 10, "type": "project"},
  {"title": "Project 4: Graph Algorithms", "due": "2026-11-20", "weight": 10, "type": "project"},
  {"title": "Midterm Exam", "due": "2026-10-28", "weight": 20, "type": "exam"},
  {"title": "Final Exam", "due": "2026-12-10", "weight": 25, "type": "exam"},
  {"title": "Weekly Quiz", "due": "2026-08-28", "weight": 0.8333333333333334, "type": "quiz"},
  {"title": "Weekly Quiz", "due": "2026-09-04", "weight": 0.8333333333333334, "type": "quiz"},
  {"title": "Weekly Quiz", "due": "2026-09-11", "weight": 0.8333333333333334, "type": "quiz"},
  {"title": "Weekly Quiz", "due": "2026-09-18", "weight": 0.8333333333333334, "type": "quiz"},
  {"title": "Weekly Quiz", "due": "2026-09-25", "weight": 0.8333333333333334, "type": "quiz"},
  {"title": "Weekly Quiz", "due": "2026-10-02", "weight": 0.8333333333333334, "type": "quiz"},
  {"title": "Weekly Quiz", "due": "2026-10-09", "weight": 0.8333333333333334, "type": "quiz"},
  {"title": "Weekly Quiz", "due": "2026-10-16", "weight": 0.8333333333333334, "type": "quiz"},
  {"title": "Weekly Quiz", "due": "2026-10-23", "weight": 0.8333333333333334, "type": "quiz"},
  {"title": "Weekly Quiz", "due": "2026-11-06", "weight": 0.8333333333333334, "type": "quiz"},
  {"title": "Weekly Quiz", "due": "2026-11-13", "weight": 0.8333333333333334, "type": "quiz"},
  {"title": "Weekly Quiz", "due": "2026-11-20", "weight": 0.8333333333333334, "type": "quiz"}
 ],
 "HIST207_Latin_America": [
  {"title": "Map Quiz", "due": "2026-09-03", "weight": 5, "type": "quiz"},
  {"title": "Midterm Essay", "due": "2026-10-27", "weight": 25, "type": "paper"},
  {"title": "Project Proposal", "due": "2026-09-25", "weight": 5, "type": "assignment"},
  {"title": "Annotated Bibliography", "due": "2026-11-13", "weight": 10, "type": "assignment"},
  {"title": "Final Paper", "due": "2026-12-11", "weight": 25, "type": "paper"},
  {"title": "Reading Response", "due": "2026-08-25", "weight": 1.6666666666666667, "type": "assignment"},
  {"title": "Reading Response", "due": "2026-09-01", "weight": 1.6666666666666667, "type": "assignment"},
  {"title": "Reading Response", "due": "2026-09-08", "weight": 1.6666666666666667, "type": "assignment"},
  {"title": "Reading Response", "due": "2026-09-15", "weight": 1.6666666666666667, "type": "assignment"},
  {"title": "Reading Response", "due": "2026-09-22", "weight": 1.6666666666666667, "type": "assignment"},
  {"title": "Reading Response", "due": "2026-09-29", "weight": 1.6666666666666667, "type": "assignment"},
  {"title": "Reading Response", "due": "2026-10-06", "weight": 1.6666666666666667, "type": "assignment"},
  {"title": "Reading Response", "due": "2026-10-13", "weight": 1.6666666666666667, "type": "assignment"},
  {"title": "Reading Response", "due": "2026-10-20", "weight": 1.6666666666666667, "type": "assignment"},
  {"title": "Reading Response", "due": "2026-10-27", "weight": 1.6666666666666667, "type": "assignment"},
  {"title": "Reading Response", "due": "2026-11-03", "weight": 1.6666666666666667, "type": "assignment"},
  {"title": "Reading Response", "due": "2026-11-10", "weight": 1.6666666666666667, "type": "assignment"}
 ]
}''')
print({k: len(v) for k, v in GROUND_TRUTH.items()}, 'deadlines')

{'CHEM101_General_Chemistry': 30, 'CSC316_Data_Structures': 18, 'HIST207_Latin_America': 17} deadlines


# Part 4: Demo

## 4.1 Read all the syllabi

In [17]:
from datetime import date
import pandas as pd
from IPython.display import display, Markdown

TODAY = date(2026, 10, 17)        # fixed date for the demo, use date.today() normally
TERM_START = date(2026, 8, 17)    # first Monday of the semester
pd.set_option("display.max_colwidth", 70)

app = SyllabusPlanner(config.SYLLABI_DIR, term_start=TERM_START, today=TODAY)
print("Mode:", app.mode, "\n")
app.run()
print(f"\n{len(app.deadlines)} dated deadlines across {len(app.reports)} courses")

Mode: LIVE (gpt-4o-mini) 

  CHEM 101   30 deadlines | 0 errors, 1 warnings | repair rounds: 0 | source: openai
  CSC 316     6 deadlines | 0 errors, 1 warnings | repair rounds: 0 | source: openai
  HIST 207   17 deadlines | 0 errors, 0 warnings | repair rounds: 0 | source: openai

53 dated deadlines across 3 courses


Here's what it got from the history syllabus (the all-paragraphs one). The dates are written the way the syllabus says them, and the 12 weekly reading responses are one repeating rule:

In [18]:
hist = app.report("HIST 207").extraction
display(pd.DataFrame([c.model_dump() for c in hist.categories]))
display(pd.DataFrame([{"title": a.title, "category": a.category,
                       "date_as_written": a.date.model_dump(exclude_none=True) if a.date else None,
                       "recurrence": a.recurrence.model_dump() if a.recurrence else None,
                       "evidence": a.evidence} for a in hist.assessments]))

,name,weight_percent,drop_lowest,evidence
0,Reading Responses,20.0,2,Reading responses make up 20% of the grade; your two lowest respon...
1,Map Quiz,5.0,0,A short map quiz counts for 5%.
2,Midterm Essay,25.0,0,The midterm essay is worth 25%.
3,Archival Research Project,40.0,0,the heart of the course and is graded in three stages: the proposa...
4,Participation,10.0,0,Participation in discussion counts for the remaining 10%.


,title,category,date_as_written,recurrence,evidence
0,Reading Responses,Reading Responses,"{'week': 2, 'weekday': 'Tue'}","{'weekday': 'Tue', 'start_week': 2, 'end_week': 13, 'skip_weeks': []}",Reading responses of about 300 words are due on Moodle before clas...
1,Map Quiz,Map Quiz,"{'week': 3, 'weekday': 'Thu'}",None,The map quiz will be given in class on the Thursday of Week 3.
2,Midterm Essay,Midterm Essay,"{'week': 11, 'weekday': 'Tue'}",None,The midterm essay prompt is released in Week 9; the essay is due o...
3,Archival Project Proposal,Archival Research Project,{'exact_date': '2026-09-25'},None,"Your one-page proposal is due Friday, September 25."
4,Annotated Bibliography,Archival Research Project,"{'week': 13, 'weekday': 'Fri'}",None,The annotated bibliography of at least eight sources is due on the...
5,Final Paper,Archival Research Project,{'exact_date': '2026-12-11'},None,"the final paper (3,000-3,500 words) is due Friday, December 11 at ..."


Results of the checks for each class. Errors get sent back to the AI, warnings just get shown:

In [19]:
for r in app.reports:
    print(f"{r.extraction.course_code}: repair rounds={r.repair_rounds}, source={r.source}")
    for i in r.issues:
        print(f"   [{i.severity}] {i.message}")
    if not r.issues:
        print("   ✓ clean")

CHEM 101: repair rounds=0, source=openai
   [warning] 'Lab Practical' has no date in the syllabus (TBA) - it is excluded from the calendar.
CSC 316: repair rounds=0, source=openai
   [warning] Category 'Weekly Quizzes' (10%) has no dated items - was something missed?
HIST 207: repair rounds=0, source=openai
   ✓ clean


## 4.2 Catching AI mistakes
To test the checker I broke one of the results on purpose: made up a quote and changed a weight so they add to 105. It catches both. When it's running with a key, these errors get sent back to the AI to fix.

In [20]:
broken = app.report("CSC 316").extraction.model_copy(deep=True)
broken.categories[1].weight_percent = 15                                       # weights now sum to 105
broken.assessments[5].evidence = "Midterm moved to Friday Oct 30 in the gym"   # not in the PDF
csc_doc = next(d for d in app.docs if d.name.startswith("CSC316"))
_, issues = resolve_and_validate(broken, csc_doc, TERM_START)
for i in issues:
    print(f"[{i.severity}] {i.message}")

[error] Grade category weights add up to 105%, not 100%. Check the grading section.
[warning] Category 'Weekly Quizzes' (15%) has no dated items - was something missed?
[error] Evidence for 'Final Exam' is not in the syllabus. Quote the syllabus verbatim or remove the item.


## 4.3 Every deadline in one table

In [21]:
df = pd.DataFrame([{"due": d.due, "day": f"{d.due:%a}", "time": d.due_time, "course": d.course,
                    "item": d.title, "type": d.type.value, "weight_%": round(d.weight_percent, 2)}
                   for d in app.deadlines])
print(f"{len(df)} deadlines. Next two weeks, worth at least 1% each:")
df[(df.due > TODAY) & (df.due <= date(2026, 10, 31)) & (df["weight_%"] >= 1)]

53 deadlines. Next two weeks, worth at least 1% each:


,due,day,time,course,item,type,weight_%
29,2026-10-20,Tue,None,HIST 207,Reading Responses #9 (Week 10),other,1.67
30,2026-10-21,Wed,23:59,CHEM 101,Lab Reports #8 (Week 10),other,1.82
31,2026-10-23,Fri,23:59,CSC 316,Project 3: Hash Maps,project,10.00
33,2026-10-27,Tue,09:00,HIST 207,Midterm Essay,other,25.00
34,2026-10-27,Tue,None,HIST 207,Reading Responses #10 (Week 11),other,1.67
35,2026-10-28,Wed,19:00,CSC 316,Midterm Exam (Weeks 1-10),exam,20.00
36,2026-10-29,Thu,None,CHEM 101,Hour Exam 2,exam,12.00


## 4.4 Accuracy
Compared to my answer key. The baseline is just using regex to find dates in the text, no AI.

In [22]:
rows = []
for doc, rep in zip(app.docs, app.reports):
    truth = load_truth(doc.name)
    if not truth:
        continue
    b = score_deadlines(baseline_extract(doc, rep.extraction.course_code), truth)
    p = score_deadlines(rep.deadlines, truth)
    rows.append({"course": rep.extraction.course_code, "truth items": len(truth),
                 "baseline recall": b["recall"], "baseline F1": b["f1"],
                 f"Planner ({rep.source}) recall": p["recall"], "Planner F1": p["f1"],
                 "weight error (pts)": p["weight_mae"]})
pd.DataFrame(rows)

,course,truth items,baseline recall,baseline F1,Planner (openai) recall,Planner F1,weight error (pts)
0,CHEM 101,30,0.033,0.065,1.000,1.0,0.000
1,CSC 316,18,0.333,0.500,0.333,0.5,0.000
2,HIST 207,17,0.059,0.105,1.000,1.0,1.373


Note: in offline mode this is scoring the saved extractions. With a key it scores what the AI actually returned.

## 4.5 Crunch weeks

In [23]:
fig = workload_chart(app.deadlines, config.OUTPUT_DIR / "workload.png", today=TODAY)
for c in crunch_weeks(app.deadlines):
    print("⚠️ ", c.describe())

⚠️  Week of Oct 26: 59% of grades due across 3 courses -> HIST 207 Midterm Essay (25%, Tue Oct 27), CSC 316 Midterm Exam (Weeks 1-10) (20%, Wed Oct 28), CHEM 101 Hour Exam 2 (12%, Thu Oct 29)
⚠️  Week of Dec 07: 38% of grades due across 2 courses -> CSC 316 Final Exam (25%, Thu Dec 10), HIST 207 Final Paper (13.333%, Fri Dec 11)


## 4.6 Study plan
Bigger assignments get more study hours (exams get extra). It spreads them out before each due date with a max of 4 hours a day, and if a week is too packed it tells you to start earlier.

In [24]:
blocks, warnings = schedule(app.deadlines, TODAY, cap=4)
for w in warnings:
    print("•", w)
summary = plan_summary(blocks, weeks=6, today=TODAY)
pd.DataFrame([{"week of": w["week_of"], "study hours": w["hours"],
               "biggest focus": "; ".join(f"{t} ({h:g}h)" for t, h in w["top"][:3])} for w in summary])

• Start CHEM 101 Hour Exam 2 today - the days before it are crowded.


,week of,study hours,biggest focus
0,2026-10-12,7.0,CSC 316 - Project 3: Hash Maps (2h); CHEM 101 - Hour Exam 2 (1.5h)...
1,2026-10-19,28.0,HIST 207 - Midterm Essay (9h); CSC 316 - Midterm Exam (Weeks 1-10)...
2,2026-10-26,13.0,CSC 316 - Midterm Exam (Weeks 1-10) (5h); CHEM 101 - Hour Exam 2 (...
3,2026-11-02,5.5,HIST 207 - Annotated Bibliography (3h); CHEM 101 - Lab Reports #9 ...
4,2026-11-09,12.5,CHEM 101 - Hour Exam 3 (5h); HIST 207 - Annotated Bibliography (3....
5,2026-11-16,10.0,CSC 316 - Project 4: Graph Algorithms (4.5h); CHEM 101 - Hour Exam...


In [25]:
plan = narrate(summary, crunch_weeks(app.deadlines), TODAY)   # AI writes this part (uses a template offline)
display(Markdown(f"**{plan.overview}**\n\n" + "\n".join(
    f"- **{w.week_of} - {w.headline}**  \n  " + " · ".join(w.focus) + (f"  \n  ⚠️ {w.risk}" if w.risk else "")
    for w in plan.weeks)))

**You're entering a crucial period with significant exams and deadlines approaching. Let's make sure you're prepared!**

- **2026-10-12 - Kick-off Crunch Time!**  
  Complete CHEM 101 - Hour Exam 2 (1.5h) preparation · Work on CSC 316 - Project 3: Hash Maps (2h) · Revise for the CSC 316 - Midterm Exam (1.5h) · Get started on the HIST 207 - Midterm Essay (1h)  
  ⚠️ With multiple exams and significant projects due soon, staying organized is key.
- **2026-10-19 - The Big Push for Midterms!**  
  Dedicate 9 hours to finalize your HIST 207 - Midterm Essay · Study 8.5 hours for the CSC 316 - Midterm Exam · Wrap up your CSC 316 - Project 3: Hash Maps (4.5h) · Prepare for CHEM 101 - Hour Exam 2 (3.5h) · Complete CHEM 101 - Lab Reports #8 (1h)  
  ⚠️ The high volume of work can lead to fatigue—take breaks and manage your time!
- **2026-10-26 - Final Sprint for Exams!**  
  Consolidate your studies for the CSC 316 - Midterm Exam (5h) · Review for CHEM 101 - Hour Exam 2 (4h) · Finalize your HIST 207 - Midterm Essay (2.5h) · Tackle HIST 207 - Reading Responses #10 (0.5h) · Finish CHEM 101 - Online Homework #11 (0.5h)  
  ⚠️ You must balance multiple deadlines—ensure your time is split effectively.
- **2026-11-02 - Focus on Research and Writing!**  
  Draft your HIST 207 - Annotated Bibliography (3h) · Complete CHEM 101 - Lab Reports #9 (1h) · Finish HIST 207 - Reading Responses #11 (0.5h) · Work on CHEM 101 - Online Homework #12 (0.5h) · Start HIST 207 - Reading Responses #12 (0.5h)  
  ⚠️ Watch out for burnout; keep a consistent work pace.
- **2026-11-09 - Midterm Review and New Projects!**  
  Prepare for CHEM 101 - Hour Exam 3 (5h) · Continue with HIST 207 - Annotated Bibliography (3.5h) · Start your CSC 316 - Project 4: Graph Algorithms (2h) · Finish CHEM 101 - Lab Reports #10 (1h) · Complete HIST 207 - Reading Responses #12 (0.5h)  
  ⚠️ With a major exam and project, ensure you have a clear study plan.
- **2026-11-16 - Final Projects and Exams Loom!**  
  Continue working on CSC 316 - Project 4: Graph Algorithms (4.5h) · Prepare extensively for CHEM 101 - Hour Exam 3 (4h) · Complete CHEM 101 - Lab Reports #11 (1h) · Finish CHEM 101 - Online Homework #14 (0.5h)  
  ⚠️ Keep an eye on remaining deadlines to avoid last-minute panic.

## 4.7 Asking questions about the syllabi (RAG)

In [26]:
for q in ["Can I turn in a CSC 316 project late?",
          "Does chem have make-up exams?",
          "What happens if I miss a reading response in history?",
          "What's the parking policy?"]:
    r = app.ask(q)
    print(f"Q: {q}\nA: {r['answer']}")
    if r["invalid_citations"]:
        print("   ⚠️ cited passages it wasn't given:", r["invalid_citations"])
    print()

Q: Can I turn in a CSC 316 project late?
A: Yes, projects may be submitted up to 48 hours late for a 10% penalty per day [CSC 316 p.1].

Q: Does chem have make-up exams?
A: There are no make-up hour exams in CHEM 101. With a documented excuse, the final exam score replaces one missed hour exam [CHEM 101 p.2].

Q: What happens if I miss a reading response in history?
A: If you miss a reading response, it won't be accepted late because the two lowest responses are dropped. This policy allows you some flexibility in completing these assignments [HIST 207 p.1].

Q: What's the parking policy?
A: The syllabus doesn't say. Please ask the instructor.



## 4.8 Ask the Planner (agent)
The agent has 5 tools: `list_deadlines`, `search_syllabus`, `grade_needed`, `crunch_weeks`, `grade_categories`. The AI picks which ones to use and Python does the actual work.

In [27]:
agent = app.agent()
questions = [
    "I have 84% on chem hour exams, 95% on labs and 100% on OWL. What do I need on the rest to get an A- (90)?",
    "What should I prioritise this week and next, and is there anything I should start early?",
]
if config.has_api_key():
    for q in questions:
        print("Q:", q)
        out = agent.ask(q)
        print("A:", out["answer"], "\n")
else:
    print("No API key, so here's what the tools return on their own:\n")
    print(agent.call_tool("grade_needed", {"course": "CHEM 101", "target_percent": 90, "scores": [
        {"category": "Hour Exams", "percent": 84}, {"category": "Lab Reports", "percent": 95},
        {"category": "OWL", "percent": 100}]}))
    for row in agent.call_tool("list_deadlines", {"start": "2026-10-17", "end": "2026-10-31", "min_weight": 5}):
        print(row)

Q: I have 84% on chem hour exams, 95% on labs and 100% on OWL. What do I need on the rest to get an A- (90)?
  -> grade_categories({"course": "CHEM 101"})
  -> grade_needed({"course": "CHEM 101", "target_percent": 90, "scores": [{"category": "Hour Exams", "percent": 84}, {"category")
A: To achieve an A- (90%) in CHEM 101, you need an average of **90.5%** on the remaining components, which account for 34% of your grade. This includes the Final Exam, Lab Practical, and Clicker Participation.

Prioritize preparing for these assessments, focusing especially on the Final Exam, as it has the highest weight (24%). Aim for at least 91% on the Final and similar scores on the other components to reach your target. Good luck! 

Q: What should I prioritise this week and next, and is there anything I should start early?
  -> list_deadlines({"course": null, "start": "2026-10-17", "end": "2026-10-30", "min_weight": 0})
  -> crunch_weeks({"from_date": "2026-10-17"})
A: This week, prioritize completing

## 4.9 What do I need to get an A?

In [28]:
chem = app.report("CHEM 101")
current = {"Hour Exams": 84, "Lab Reports": 95, "OWL": 100}
pd.DataFrame([{"target": f"{letter} ({cut}%)",
               **{k: v for k, v in grade_needed(chem, current, cut).items() if k in ("needed_average", "note")}}
              for letter, cut in LETTERS[:6]])

,target,needed_average,note
0,A (93%),99.3,You need an average of 99.3% on the remaining 34% of the grade.
1,A- (90%),90.5,You need an average of 90.5% on the remaining 34% of the grade.
2,B+ (87%),81.6,You need an average of 81.6% on the remaining 34% of the grade.
3,B (83%),69.9,You need an average of 69.9% on the remaining 34% of the grade.
4,B- (80%),61.1,You need an average of 61.1% on the remaining 34% of the grade.
5,C+ (77%),52.2,You need an average of 52.2% on the remaining 34% of the grade.


## 4.10 Export to calendar

In [29]:
ics = to_ics(app.deadlines, config.OUTPUT_DIR / "semester.ics", blocks)
csv_path = plan_to_csv(blocks, config.OUTPUT_DIR / "study_plan.csv")
print("Calendar file saved:", ics)
print("To use it: download it from the outputs folder, then in Google Calendar go to Settings > Import & export > Import")
print("Study plan:", csv_path)

Calendar file saved: /content/outputs/semester.ics
To use it: download it from the outputs folder, then in Google Calendar go to Settings > Import & export > Import
Study plan: /content/outputs/study_plan.csv


## 4.11 Try your own syllabi (optional, needs a key)
Put your syllabus PDFs in a folder, set `MY_DIR` to that folder, and set `TERM_START` to the Monday of your first week.

In [30]:
MY_DIR = None   # e.g. "my_syllabi"
if MY_DIR:
    mine = SyllabusPlanner(MY_DIR, term_start=TERM_START, today=date.today()).run()
    display(pd.DataFrame([d.model_dump(include={"due", "course", "title", "weight_percent"}) for d in mine.deadlines]))
else:
    print("Skipped, set MY_DIR to use your own syllabi.")

Skipped, set MY_DIR to use your own syllabi.


# Part 5: Tests
13 quick tests to make sure everything actually works. None of them need an API key. For the fix-it loop and the agent, I swap in a fake AI that gives set answers, so the test does the same thing every time.

| # | What it tests | Why it matters |
|---|---|---|
| 1-3 | For each sample syllabus, the dates and weights match my hand-made answer key exactly | proves the date conversion and weight splitting are right |
| 4 | Week 11 Tuesday comes out as Oct 27 | checks the week-to-date math |
| 5 | A real quote from the syllabus passes the quote check | the checker doesn't reject correct answers |
| 6 | A made-up quote fails the quote check | the checker catches the AI making stuff up |
| 7 | When the fake AI gives a bad answer (weights add to 105, fake quote), the errors get sent back to it | the fix-it loop sends the right feedback |
| 8 | After the fake AI fixes it, there are no errors left and it took 1 round | the fix-it loop actually fixes things |
| 9 | The agent calls a tool and uses the result in its answer | the agent loop works |
| 10 | Giving a tool bad input (like a date that isn't a date) returns an error message instead of crashing | the agent can't break the notebook |
| 11 | The study plan never has more than 4 hours in a day | the daily limit works |
| 12 | The grade calculator says you need 90.5% for an A- in chem (with 84 / 95 / 100 so far) | the grade math is right |
| 13 | The week of Oct 26 gets flagged as a crunch week | crunch detection works |

In [31]:
from types import SimpleNamespace

checks = []
def check(name, ok):
    checks.append((name, bool(ok)))
    print(("✅" if ok else "❌"), f"{len(checks)}.", name)

def ref_report(doc):
    return analyze_reference(doc, TERM_START)

docs = {d.name: d for d in app.docs}

# dates and weights should match my answer key exactly
for name, doc in docs.items():
    s = score_deadlines(ref_report(doc).deadlines, load_truth(name))
    check(f"dates and weights match answer key: {name}", s["f1"] == 1.0 and s["weight_mae"] == 0.0)

# week math
check("week 11 Tuesday comes out as Oct 27", week_day(TERM_START, 11, "Tue") == date(2026, 10, 27))

# real quote should pass, fake one should fail
hist_norm = _norm(docs["HIST207_Latin_America"].text)
check("real quote passes the quote check", grounded("the essay is due on the Tuesday of Week 11 by 9:00 AM", hist_norm))
check("made-up quote fails the quote check", not grounded("The final exam is worth 30% and is held in Week 17", hist_norm))

# fix-it loop: fake AI gives a bad answer first, then a good one
good = CourseExtraction.model_validate(REFERENCE_EXTRACTIONS["HIST207_Latin_America"])
bad = good.model_copy(deep=True)
bad.categories[2].weight_percent = 30
bad.assessments[1].evidence = "Map quiz is on Friday of Week 2"
calls, real_parse = [], llm_parse
def fake_parse(messages, schema, model=None):
    calls.append(messages)
    return bad if len(calls) == 1 else good
llm_parse = fake_parse
try:
    rep = analyze_syllabus(docs["HIST207_Latin_America"], TERM_START, use_llm=True)
finally:
    llm_parse = real_parse
feedback = calls[1][-1]["content"] if len(calls) > 1 else ""
check("fix-it loop: errors get sent back to the AI", "105" in feedback and "Map Quiz" in feedback)
check("fix-it loop: no errors left after 1 round", rep.repair_rounds == 1 and not [i for i in rep.issues if i.severity == "error"])

# agent: fake AI calls a tool and then answers using the result
seen, real_chat, real_key = [], llm_chat, Config.has_api_key
def fake_chat(messages, tools=None, model=None):
    seen.append(messages)
    if len(seen) == 1:
        tc = SimpleNamespace(id="call_1", function=SimpleNamespace(
            name="list_deadlines", arguments=json.dumps({"start": "2026-10-26", "end": "2026-10-30", "min_weight": 10})))
        return SimpleNamespace(content=None, tool_calls=[tc])
    titles = ", ".join(r["title"] for r in json.loads(messages[-1]["content"]))
    return SimpleNamespace(content=f"Big items: {titles}", tool_calls=None)
llm_chat, Config.has_api_key = fake_chat, staticmethod(lambda: True)
try:
    out = app.agent().ask("what's big next week?", verbose=False)
finally:
    llm_chat, Config.has_api_key = real_chat, real_key
check("agent calls a tool and uses the result", out["trace"][0]["tool"] == "list_deadlines" and "Midterm Essay" in out["answer"])
check("bad tool input returns an error instead of crashing", "error" in app.agent().call_tool("list_deadlines", {"start": "not-a-date"}))

# planner
blocks_chk, _ = schedule(app.deadlines, TODAY, cap=4)
per_day = {}
for b in blocks_chk:
    per_day[b.day] = per_day.get(b.day, 0) + b.hours
check("study plan never goes over 4 hours in a day", max(per_day.values()) <= 4)
check("grade calculator: A- in chem needs 90.5%",
      grade_needed(app.report("CHEM 101"), {"Hour Exams": 84, "Lab": 95, "OWL": 100}, 90)["needed_average"] == 90.5)
check("week of Oct 26 is flagged as a crunch week", date(2026, 10, 26) in [c.week_of for c in crunch_weeks(app.deadlines)])

print(f"\n{sum(ok for _, ok in checks)}/{len(checks)} tests passed")

✅ 1. dates and weights match answer key: CHEM101_General_Chemistry
✅ 2. dates and weights match answer key: CSC316_Data_Structures
✅ 3. dates and weights match answer key: HIST207_Latin_America
✅ 4. week 11 Tuesday comes out as Oct 27
✅ 5. real quote passes the quote check
✅ 6. made-up quote fails the quote check
✅ 7. fix-it loop: errors get sent back to the AI
✅ 8. fix-it loop: no errors left after 1 round
✅ 9. agent calls a tool and uses the result
✅ 10. bad tool input returns an error instead of crashing
✅ 11. study plan never goes over 4 hours in a day
✅ 12. grade calculator: A- in chem needs 90.5%
✅ 13. week of Oct 26 is flagged as a crunch week

13/13 tests passed
